In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "e3bf2c11",
   "metadata": {},
   "source": [
    "# Case Study 3 — Predicting Titanic Survival with Logistic Regression\n",
    "\n",
    "| | |\n",
    "|---|---|\n",
    "| **Subject** | Fundamentals of Data Science |\n",
    "| **Course code** | 22CSC59 |\n",
    "| **Faculty** | S. Durga Devi |\n",
    "\n",
    "**Problem statement.** Given passenger records from the RMS Titanic (891 passengers, 12 attributes), build a\n",
    "logistic regression model that estimates each passenger's probability of survival, evaluate it on held-out\n",
    "data (accuracy, precision, recall, F1, ROC-AUC), interpret which factors drove survival, and deploy the\n",
    "model as a Streamlit web app.\n",
    "\n",
    "**Why logistic regression.** The target is binary (survived / did not survive). Logistic regression models the\n",
    "*log-odds* of survival as a linear combination of features, so every coefficient has a direct meaning\n",
    "(\"being male multiplies the odds of survival by X\"). That interpretability is the point of this case study —\n",
    "a black-box model could score slightly higher but could not explain *why*.\n",
    "\n",
    "**Notebook outline**\n",
    "1. Data exploration (structure, summary statistics, visualisations, statistical tests)\n",
    "2. Data preprocessing (missing values, encoding, scaling — inside a leakage-safe pipeline)\n",
    "3. Model building and training\n",
    "4. Model evaluation (metrics, confusion matrix, ROC curve, cross-validation)\n",
    "5. Interpretation (coefficients, odds ratios, statistical significance)\n",
    "6. Saving the model for the Streamlit app (`app.py`)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a8586bd9",
   "metadata": {},
   "source": [
    "## 0. Setup\n",
    "Import the libraries used throughout. `RANDOM_STATE` is fixed so that the train/test split and every\n",
    "result in this notebook is reproducible — re-running gives exactly the same numbers quoted in the report."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "f1576a27",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:48.715739Z",
     "iopub.status.busy": "2026-10-25T17:02:48.715522Z",
     "iopub.status.idle": "2026-10-25T17:02:57.699500Z",
     "shell.execute_reply": "2026-10-25T17:02:57.698477Z"
    }
   },
   "outputs": [],
   "source": [
    "import os, warnings\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from scipy import stats\n",
    "import statsmodels.api as sm\n",
    "import joblib\n",
    "\n",
    "from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.compose import ColumnTransformer\n",
    "from sklearn.preprocessing import OneHotEncoder, StandardScaler\n",
    "from sklearn.impute import SimpleImputer\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,\n",
    "                             roc_auc_score, roc_curve, confusion_matrix, classification_report,\n",
    "                             ConfusionMatrixDisplay)\n",
    "\n",
    "warnings.filterwarnings(\"ignore\")\n",
    "RANDOM_STATE = 42\n",
    "sns.set_theme(style=\"whitegrid\", context=\"notebook\")\n",
    "os.makedirs(\"figures\", exist_ok=True)\n",
    "os.makedirs(\"model\", exist_ok=True)\n",
    "\n",
    "def savefig(name):\n",
    "    plt.tight_layout()\n",
    "    plt.savefig(f\"figures/{name}.png\", dpi=150, bbox_inches=\"tight\")\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "32267a78",
   "metadata": {},
   "source": [
    "## 1. Data Exploration\n",
    "### 1a. Load the dataset\n",
    "**Purpose:** read the CSV into a DataFrame and look at the first rows to understand what each record represents."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "22b67797",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:57.706784Z",
     "iopub.status.busy": "2026-10-25T17:02:57.706395Z",
     "iopub.status.idle": "2026-10-25T17:02:57.794019Z",
     "shell.execute_reply": "2026-10-25T17:02:57.792663Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Shape: (891, 12)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>PassengerId</th>\n",
       "      <th>Survived</th>\n",
       "      <th>Pclass</th>\n",
       "      <th>Name</th>\n",
       "      <th>Sex</th>\n",
       "      <th>Age</th>\n",
       "      <th>SibSp</th>\n",
       "      <th>Parch</th>\n",
       "      <th>Ticket</th>\n",
       "      <th>Fare</th>\n",
       "      <th>Cabin</th>\n",
       "      <th>Embarked</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "      <td>3</td>\n",
       "      <td>Braund, Mr. Owen Harris</td>\n",
       "      <td>male</td>\n",
       "      <td>22.0</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "      <td>A/5 21171</td>\n",
       "      <td>7.2500</td>\n",
       "      <td>NaN</td>\n",
       "      <td>S</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>2</td>\n",
       "      <td>1</td>\n",
       "      <td>1</td>\n",
       "      <td>Cumings, Mrs. John Bradley (Florence Briggs Th...</td>\n",
       "      <td>female</td>\n",
       "      <td>38.0</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "      <td>PC 17599</td>\n",
       "      <td>71.2833</td>\n",
       "      <td>C85</td>\n",
       "      <td>C</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>3</td>\n",
       "      <td>1</td>\n",
       "      <td>3</td>\n",
       "      <td>Heikkinen, Miss. Laina</td>\n",
       "      <td>female</td>\n",
       "      <td>26.0</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>STON/O2. 3101282</td>\n",
       "      <td>7.9250</td>\n",
       "      <td>NaN</td>\n",
       "      <td>S</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>4</td>\n",
       "      <td>1</td>\n",
       "      <td>1</td>\n",
       "      <td>Futrelle, Mrs. Jacques Heath (Lily May Peel)</td>\n",
       "      <td>female</td>\n",
       "      <td>35.0</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "      <td>113803</td>\n",
       "      <td>53.1000</td>\n",
       "      <td>C123</td>\n",
       "      <td>S</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>5</td>\n",
       "      <td>0</td>\n",
       "      <td>3</td>\n",
       "      <td>Allen, Mr. William Henry</td>\n",
       "      <td>male</td>\n",
       "      <td>35.0</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>373450</td>\n",
       "      <td>8.0500</td>\n",
       "      <td>NaN</td>\n",
       "      <td>S</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "   PassengerId  Survived  Pclass  \\\n",
       "0            1         0       3   \n",
       "1            2         1       1   \n",
       "2            3         1       3   \n",
       "3            4         1       1   \n",
       "4            5         0       3   \n",
       "\n",
       "                                                Name     Sex   Age  SibSp  \\\n",
       "0                            Braund, Mr. Owen Harris    male  22.0      1   \n",
       "1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   \n",
       "2                             Heikkinen, Miss. Laina  female  26.0      0   \n",
       "3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   \n",
       "4                           Allen, Mr. William Henry    male  35.0      0   \n",
       "\n",
       "   Parch            Ticket     Fare Cabin Embarked  \n",
       "0      0         A/5 21171   7.2500   NaN        S  \n",
       "1      0          PC 17599  71.2833   C85        C  \n",
       "2      0  STON/O2. 3101282   7.9250   NaN        S  \n",
       "3      0            113803  53.1000  C123        S  \n",
       "4      0            373450   8.0500   NaN        S  "
      ]
     },
     "execution_count": 2,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df = pd.read_csv(\"data/titanic.csv\")\n",
    "print(\"Shape:\", df.shape)\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "254b37b0",
   "metadata": {},
   "source": [
    "**Observation:** each row is one passenger. `Survived` (0 = died, 1 = survived) is the target. `PassengerId`,\n",
    "`Name` and `Ticket` are identifiers rather than predictors; `Cabin` is mostly empty (see below)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "88f3e536",
   "metadata": {},
   "source": [
    "### 1b. Features, data types and summary statistics\n",
    "**Purpose:** confirm each column's type, count missing values, and get the central tendency / spread of the\n",
    "numeric columns."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "0b4edbba",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:57.796651Z",
     "iopub.status.busy": "2026-10-25T17:02:57.796336Z",
     "iopub.status.idle": "2026-10-25T17:02:57.815604Z",
     "shell.execute_reply": "2026-10-25T17:02:57.814573Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>dtype</th>\n",
       "      <th>non_null</th>\n",
       "      <th>missing</th>\n",
       "      <th>missing_%</th>\n",
       "      <th>unique</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>PassengerId</th>\n",
       "      <td>int64</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>891</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Survived</th>\n",
       "      <td>int64</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>2</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Pclass</th>\n",
       "      <td>int64</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>3</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Name</th>\n",
       "      <td>str</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>891</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Sex</th>\n",
       "      <td>str</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>2</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Age</th>\n",
       "      <td>float64</td>\n",
       "      <td>714</td>\n",
       "      <td>177</td>\n",
       "      <td>19.9</td>\n",
       "      <td>88</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>SibSp</th>\n",
       "      <td>int64</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>7</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Parch</th>\n",
       "      <td>int64</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>7</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Ticket</th>\n",
       "      <td>str</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>681</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Fare</th>\n",
       "      <td>float64</td>\n",
       "      <td>891</td>\n",
       "      <td>0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>248</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Cabin</th>\n",
       "      <td>str</td>\n",
       "      <td>204</td>\n",
       "      <td>687</td>\n",
       "      <td>77.1</td>\n",
       "      <td>147</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Embarked</th>\n",
       "      <td>str</td>\n",
       "      <td>889</td>\n",
       "      <td>2</td>\n",
       "      <td>0.2</td>\n",
       "      <td>3</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "               dtype  non_null  missing  missing_%  unique\n",
       "PassengerId    int64       891        0        0.0     891\n",
       "Survived       int64       891        0        0.0       2\n",
       "Pclass         int64       891        0        0.0       3\n",
       "Name             str       891        0        0.0     891\n",
       "Sex              str       891        0        0.0       2\n",
       "Age          float64       714      177       19.9      88\n",
       "SibSp          int64       891        0        0.0       7\n",
       "Parch          int64       891        0        0.0       7\n",
       "Ticket           str       891        0        0.0     681\n",
       "Fare         float64       891        0        0.0     248\n",
       "Cabin            str       204      687       77.1     147\n",
       "Embarked         str       889        2        0.2       3"
      ]
     },
     "execution_count": 3,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "feature_info = pd.DataFrame({\n",
    "    \"dtype\": df.dtypes.astype(str),\n",
    "    \"non_null\": df.notna().sum(),\n",
    "    \"missing\": df.isna().sum(),\n",
    "    \"missing_%\": (df.isna().mean() * 100).round(1),\n",
    "    \"unique\": df.nunique(),\n",
    "})\n",
    "feature_info"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "a934fe3d",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:57.817623Z",
     "iopub.status.busy": "2026-10-25T17:02:57.817404Z",
     "iopub.status.idle": "2026-10-25T17:02:57.850309Z",
     "shell.execute_reply": "2026-10-25T17:02:57.849090Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>PassengerId</th>\n",
       "      <th>Survived</th>\n",
       "      <th>Pclass</th>\n",
       "      <th>Age</th>\n",
       "      <th>SibSp</th>\n",
       "      <th>Parch</th>\n",
       "      <th>Fare</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>count</th>\n",
       "      <td>891.00</td>\n",
       "      <td>891.00</td>\n",
       "      <td>891.00</td>\n",
       "      <td>714.00</td>\n",
       "      <td>891.00</td>\n",
       "      <td>891.00</td>\n",
       "      <td>891.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>mean</th>\n",
       "      <td>446.00</td>\n",
       "      <td>0.38</td>\n",
       "      <td>2.31</td>\n",
       "      <td>29.70</td>\n",
       "      <td>0.52</td>\n",
       "      <td>0.38</td>\n",
       "      <td>32.20</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>std</th>\n",
       "      <td>257.35</td>\n",
       "      <td>0.49</td>\n",
       "      <td>0.84</td>\n",
       "      <td>14.53</td>\n",
       "      <td>1.10</td>\n",
       "      <td>0.81</td>\n",
       "      <td>49.69</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>min</th>\n",
       "      <td>1.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>1.00</td>\n",
       "      <td>0.42</td>\n",
       "      <td>0.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>0.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>25%</th>\n",
       "      <td>223.50</td>\n",
       "      <td>0.00</td>\n",
       "      <td>2.00</td>\n",
       "      <td>20.12</td>\n",
       "      <td>0.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>7.91</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>50%</th>\n",
       "      <td>446.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>3.00</td>\n",
       "      <td>28.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>14.45</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>75%</th>\n",
       "      <td>668.50</td>\n",
       "      <td>1.00</td>\n",
       "      <td>3.00</td>\n",
       "      <td>38.00</td>\n",
       "      <td>1.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>31.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>max</th>\n",
       "      <td>891.00</td>\n",
       "      <td>1.00</td>\n",
       "      <td>3.00</td>\n",
       "      <td>80.00</td>\n",
       "      <td>8.00</td>\n",
       "      <td>6.00</td>\n",
       "      <td>512.33</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "       PassengerId  Survived  Pclass     Age   SibSp   Parch    Fare\n",
       "count       891.00    891.00  891.00  714.00  891.00  891.00  891.00\n",
       "mean        446.00      0.38    2.31   29.70    0.52    0.38   32.20\n",
       "std         257.35      0.49    0.84   14.53    1.10    0.81   49.69\n",
       "min           1.00      0.00    1.00    0.42    0.00    0.00    0.00\n",
       "25%         223.50      0.00    2.00   20.12    0.00    0.00    7.91\n",
       "50%         446.00      0.00    3.00   28.00    0.00    0.00   14.45\n",
       "75%         668.50      1.00    3.00   38.00    1.00    0.00   31.00\n",
       "max         891.00      1.00    3.00   80.00    8.00    6.00  512.33"
      ]
     },
     "execution_count": 4,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df.describe().round(2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "c23a8007",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:57.852993Z",
     "iopub.status.busy": "2026-10-25T17:02:57.852772Z",
     "iopub.status.idle": "2026-10-25T17:02:57.870865Z",
     "shell.execute_reply": "2026-10-25T17:02:57.869925Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Name</th>\n",
       "      <th>Sex</th>\n",
       "      <th>Ticket</th>\n",
       "      <th>Cabin</th>\n",
       "      <th>Embarked</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>count</th>\n",
       "      <td>891</td>\n",
       "      <td>891</td>\n",
       "      <td>891</td>\n",
       "      <td>204</td>\n",
       "      <td>889</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>unique</th>\n",
       "      <td>891</td>\n",
       "      <td>2</td>\n",
       "      <td>681</td>\n",
       "      <td>147</td>\n",
       "      <td>3</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>top</th>\n",
       "      <td>Braund, Mr. Owen Harris</td>\n",
       "      <td>male</td>\n",
       "      <td>347082</td>\n",
       "      <td>G6</td>\n",
       "      <td>S</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>freq</th>\n",
       "      <td>1</td>\n",
       "      <td>577</td>\n",
       "      <td>7</td>\n",
       "      <td>4</td>\n",
       "      <td>644</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                           Name   Sex  Ticket Cabin Embarked\n",
       "count                       891   891     891   204      889\n",
       "unique                      891     2     681   147        3\n",
       "top     Braund, Mr. Owen Harris  male  347082    G6        S\n",
       "freq                          1   577       7     4      644"
      ]
     },
     "execution_count": 5,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df.describe(include=\"object\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "986f6156",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:57.875008Z",
     "iopub.status.busy": "2026-10-25T17:02:57.874801Z",
     "iopub.status.idle": "2026-10-25T17:02:57.888399Z",
     "shell.execute_reply": "2026-10-25T17:02:57.887135Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Duplicate rows: 0\n",
      "\n",
      "Target distribution:\n",
      "Survived\n",
      "Died (0)        549\n",
      "Survived (1)    342\n",
      "Name: count, dtype: int64\n",
      "Survived\n",
      "Died %        61.62\n",
      "Survived %    38.38\n",
      "Name: proportion, dtype: float64\n"
     ]
    }
   ],
   "source": [
    "print(\"Duplicate rows:\", df.duplicated().sum())\n",
    "print(\"\\nTarget distribution:\")\n",
    "print(df[\"Survived\"].value_counts().rename({0: \"Died (0)\", 1: \"Survived (1)\"}))\n",
    "print((df[\"Survived\"].value_counts(normalize=True) * 100).round(2).rename({0: \"Died %\", 1: \"Survived %\"}))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "495fc136",
   "metadata": {},
   "source": [
    "**Observations**\n",
    "- **Feature types.** Numeric: `Age`, `Fare`, `SibSp` (siblings/spouses aboard), `Parch` (parents/children aboard).\n",
    "  Categorical: `Sex`, `Embarked` (port: C = Cherbourg, Q = Queenstown, S = Southampton), and `Pclass`\n",
    "  (1/2/3 — stored as an integer but it is an *ordered category*, so it will be one-hot encoded).\n",
    "- **Missing data.** `Cabin` 77.1% missing, `Age` 19.9% (177 rows), `Embarked` 2 rows. No duplicate rows.\n",
    "- **Target balance.** 61.6% died, 38.4% survived — mildly imbalanced. A model that always predicts \"died\" would\n",
    "  already score ~62% accuracy, so accuracy alone is a weak yardstick; precision, recall, F1 and ROC-AUC are reported too.\n",
    "- **Fare is heavily right-skewed** (mean 32.2 vs median 14.45, max 512.3; skewness ≈ 4.8). A log transform will be applied."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "77caf91b",
   "metadata": {},
   "source": [
    "### 1c. Visualisations\n",
    "#### Missing values per column\n",
    "**Purpose:** decide how to treat each column with gaps."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "f22fb933",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:57.890628Z",
     "iopub.status.busy": "2026-10-25T17:02:57.890397Z",
     "iopub.status.idle": "2026-10-25T17:02:58.131393Z",
     "shell.execute_reply": "2026-10-25T17:02:58.130017Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 700x350 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "miss = df.isna().mean().mul(100).sort_values(ascending=False)\n",
    "miss = miss[miss > 0]\n",
    "fig, ax = plt.subplots(figsize=(7, 3.5))\n",
    "bars = ax.bar(miss.index, miss.values, color=[\"#c0392b\", \"#e67e22\", \"#f1c40f\"])\n",
    "ax.bar_label(bars, fmt=\"%.1f%%\")\n",
    "ax.set_title(\"Percentage of Missing Values by Column\")\n",
    "ax.set_xlabel(\"Column\")\n",
    "ax.set_ylabel(\"Missing values (%)\")\n",
    "ax.set_ylim(0, 90)\n",
    "savefig(\"cs3_01_missing\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fd3dfa26",
   "metadata": {},
   "source": [
    "**Interpretation:** `Cabin` is too sparse (77%) to impute meaningfully, so the raw value is dropped — but *whether*\n",
    "a cabin was recorded is itself informative (cabins were recorded mostly for 1st-class passengers), so it is kept as a\n",
    "binary `HasCabin` flag. `Age` (20% missing) is imputed with the training-set median and a flag `AgeMissing` records\n",
    "that it was imputed. `Embarked` (0.2%) is filled with the most frequent port."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5573dc3b",
   "metadata": {},
   "source": [
    "#### Target distribution\n",
    "**Purpose:** see the class balance the model must learn from."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "7aa8b849",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:58.133596Z",
     "iopub.status.busy": "2026-10-25T17:02:58.133357Z",
     "iopub.status.idle": "2026-10-25T17:02:58.328160Z",
     "shell.execute_reply": "2026-10-25T17:02:58.327111Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 600x400 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "counts = df[\"Survived\"].value_counts().sort_index()\n",
    "fig, ax = plt.subplots(figsize=(6, 4))\n",
    "bars = ax.bar([\"Did not survive (0)\", \"Survived (1)\"], counts.values, color=[\"#7f8c8d\", \"#27ae60\"])\n",
    "ax.bar_label(bars, labels=[f\"{c} ({c/len(df):.1%})\" for c in counts.values])\n",
    "ax.set_title(\"Distribution of the Target Variable (Survived)\")\n",
    "ax.set_xlabel(\"Outcome\")\n",
    "ax.set_ylabel(\"Number of passengers\")\n",
    "savefig(\"cs3_02_target\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2ea8fe64",
   "metadata": {},
   "source": [
    "**Interpretation:** 549 passengers died and 342 survived (38.4%). The imbalance is moderate, so a stratified\n",
    "train/test split is used to keep the same 62/38 ratio in both sets."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "efeb9b25",
   "metadata": {},
   "source": [
    "#### Survival rate by categorical features\n",
    "**Purpose:** compare survival across sex, class and port of embarkation."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "8608e82d",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:58.330392Z",
     "iopub.status.busy": "2026-10-25T17:02:58.330191Z",
     "iopub.status.idle": "2026-10-25T17:02:58.979176Z",
     "shell.execute_reply": "2026-10-25T17:02:58.978326Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1500x420 with 3 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)\n",
    "for ax, col, labels in zip(axes, [\"Sex\", \"Pclass\", \"Embarked\"],\n",
    "                           [None, {1: \"1st\", 2: \"2nd\", 3: \"3rd\"}, {\"C\": \"Cherbourg\", \"Q\": \"Queenstown\", \"S\": \"Southampton\"}]):\n",
    "    rate = df.groupby(col)[\"Survived\"].mean()\n",
    "    n = df.groupby(col).size()\n",
    "    names = [labels.get(i, i) if labels else str(i).title() for i in rate.index]\n",
    "    bars = ax.bar(names, rate.values * 100, color=sns.color_palette(\"Set2\", len(rate)))\n",
    "    ax.bar_label(bars, labels=[f\"{r:.1%}\\n(n={k})\" for r, k in zip(rate.values, n.values)], padding=8)\n",
    "    ax.axhline(df[\"Survived\"].mean() * 100, ls=\"--\", c=\"black\", lw=1, label=\"Overall 38.4%\")\n",
    "    ax.set_title(f\"Survival Rate by {col}\")\n",
    "    ax.set_xlabel(col)\n",
    "    ax.set_ylim(0, 100)\n",
    "axes[0].set_ylabel(\"Survival rate (%)\")\n",
    "axes[0].legend(loc=\"upper right\")\n",
    "savefig(\"cs3_03_survival_by_category\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1d657ddb",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Sex is the strongest single signal:** 74.2% of women survived versus 18.9% of men — consistent with the\n",
    "  \"women and children first\" evacuation order.\n",
    "- **Class shows a clear gradient:** 63.0% (1st) → 47.3% (2nd) → 24.2% (3rd). Higher decks were closer to the\n",
    "  lifeboats and 1st-class passengers were alerted earlier.\n",
    "- **Port matters less directly:** Cherbourg passengers survived at 55.4% vs 33.7% for Southampton. This is largely\n",
    "  a *class* effect — a larger share of Cherbourg passengers travelled 1st class — which the multivariate model will separate out."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0cca3126",
   "metadata": {},
   "source": [
    "#### Interaction of sex and class\n",
    "**Purpose:** check whether the sex effect is the same in every class."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "ded45e54",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:58.982092Z",
     "iopub.status.busy": "2026-10-25T17:02:58.981819Z",
     "iopub.status.idle": "2026-10-25T17:02:59.279376Z",
     "shell.execute_reply": "2026-10-25T17:02:59.277867Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 600x400 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "pivot = df.pivot_table(index=\"Pclass\", columns=\"Sex\", values=\"Survived\", aggfunc=\"mean\") * 100\n",
    "fig, ax = plt.subplots(figsize=(6, 4))\n",
    "sns.heatmap(pivot, annot=True, fmt=\".1f\", cmap=\"RdYlGn\", vmin=0, vmax=100,\n",
    "            cbar_kws={\"label\": \"Survival rate (%)\"}, ax=ax)\n",
    "ax.set_title(\"Survival Rate (%) by Passenger Class and Sex\")\n",
    "ax.set_xlabel(\"Sex\")\n",
    "ax.set_ylabel(\"Passenger class\")\n",
    "savefig(\"cs3_04_class_sex_heatmap\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e8fcfe1e",
   "metadata": {},
   "source": [
    "**Interpretation:** 1st- and 2nd-class women survived at 96.8% and 92.1%, but 3rd-class women only at 50.0%.\n",
    "Men in 2nd and 3rd class fared worst (15.7% and 13.5%). So sex and class act *together*: being female helped\n",
    "everywhere, but far less so in 3rd class."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "746f1679",
   "metadata": {},
   "source": [
    "#### Distributions of the numeric features — histograms\n",
    "**Purpose:** compare the shape of `Age` and `Fare` for survivors and non-survivors."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "b2da817f",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:02:59.282167Z",
     "iopub.status.busy": "2026-10-25T17:02:59.281867Z",
     "iopub.status.idle": "2026-10-25T17:03:00.133631Z",
     "shell.execute_reply": "2026-10-25T17:03:00.132379Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x450 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "from matplotlib.patches import Patch\n",
    "pal = {0: \"#7f8c8d\", 1: \"#27ae60\"}\n",
    "handles = [Patch(color=pal[0], alpha=0.6, label=\"Did not survive (0)\"),\n",
    "           Patch(color=pal[1], alpha=0.6, label=\"Survived (1)\")]\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))\n",
    "sns.histplot(data=df, x=\"Age\", hue=\"Survived\", bins=30, kde=True, ax=axes[0],\n",
    "             palette=pal, alpha=0.5, legend=False)\n",
    "axes[0].set_title(\"Age Distribution by Survival\")\n",
    "axes[0].set_xlabel(\"Age (years)\")\n",
    "axes[0].set_ylabel(\"Number of passengers\")\n",
    "axes[0].legend(handles=handles, title=\"Outcome\")\n",
    "\n",
    "tmp = df.assign(LogFare=np.log1p(df[\"Fare\"]))\n",
    "sns.histplot(data=tmp, x=\"LogFare\", hue=\"Survived\", bins=30, kde=True, ax=axes[1],\n",
    "             palette=pal, alpha=0.5, legend=False)\n",
    "ticks = [0, 5, 10, 25, 50, 100, 250, 500]\n",
    "axes[1].set_xticks(np.log1p(ticks))\n",
    "axes[1].set_xticklabels([f\"£{t}\" for t in ticks])\n",
    "axes[1].set_title(\"Fare Distribution by Survival (log-scaled fare axis)\")\n",
    "axes[1].set_xlabel(\"Fare paid (£, log scale)\")\n",
    "axes[1].set_ylabel(\"Number of passengers\")\n",
    "axes[1].legend(handles=handles, title=\"Outcome\")\n",
    "savefig(\"cs3_05_histograms\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f8dd4b89",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Age:** the two distributions overlap heavily, except at the youngest ages, where survivors dominate\n",
    "  (children under 13 survived at 58.0% vs 36.7% for everyone else). Most passengers were 20–40.\n",
    "- **Fare:** on a log scale the picture is clear — non-survivors cluster at cheap fares (about £7–£15, typical 3rd-class\n",
    "  tickets), while survivors are spread towards higher fares and outnumber non-survivors above roughly £50. Raw fare has a\n",
    "  long right tail (skewness 4.8, max £512), which is why `log(1 + Fare)` is used in the model (skewness drops to 0.39)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6b3da6cb",
   "metadata": {},
   "source": [
    "#### Box plots\n",
    "**Purpose:** compare medians and spread, and spot outliers."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "999dc550",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:00.135724Z",
     "iopub.status.busy": "2026-10-25T17:03:00.135542Z",
     "iopub.status.idle": "2026-10-25T17:03:00.794359Z",
     "shell.execute_reply": "2026-10-25T17:03:00.793182Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x450 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))\n",
    "sns.boxplot(data=df, x=\"Survived\", y=\"Age\", ax=axes[0], palette=[\"#7f8c8d\", \"#27ae60\"])\n",
    "axes[0].set_xticklabels([\"Did not survive\", \"Survived\"])\n",
    "axes[0].set_title(\"Age by Survival Outcome\")\n",
    "axes[0].set_xlabel(\"Outcome\")\n",
    "axes[0].set_ylabel(\"Age (years)\")\n",
    "\n",
    "sns.boxplot(data=df, x=\"Pclass\", y=\"Fare\", hue=\"Survived\", ax=axes[1], palette=[\"#7f8c8d\", \"#27ae60\"])\n",
    "axes[1].set_yscale(\"log\")\n",
    "axes[1].set_title(\"Fare by Passenger Class and Survival (log scale)\")\n",
    "axes[1].set_xlabel(\"Passenger class\")\n",
    "axes[1].set_ylabel(\"Fare (£, log scale)\")\n",
    "h, _ = axes[1].get_legend_handles_labels()\n",
    "axes[1].legend(h, [\"Did not survive\", \"Survived\"], title=\"Outcome\")\n",
    "savefig(\"cs3_06_boxplots\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e058d7c1",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Age:** median age is 28 for both groups — the box plot confirms age on its own barely separates outcomes.\n",
    "- **Fare by class:** fare rises steeply with class (it is strongly tied to `Pclass`, correlation −0.55). *Within*\n",
    "  1st and 2nd class, survivors' median fare is clearly higher than non-survivors' (in 3rd class the medians are almost\n",
    "  equal), suggesting fare carries some information beyond class — e.g., better cabins closer to the boat deck. The many high-fare points above the whiskers are genuine luxury\n",
    "  tickets, not data errors, so they are kept (the log transform tames their influence)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "ca54a606",
   "metadata": {},
   "source": [
    "#### Pair plot\n",
    "**Purpose:** see pairwise relationships between numeric features, coloured by outcome."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "id": "5b2b3dec",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:00.796356Z",
     "iopub.status.busy": "2026-10-25T17:03:00.796180Z",
     "iopub.status.idle": "2026-10-25T17:03:03.235667Z",
     "shell.execute_reply": "2026-10-25T17:03:03.233837Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1005.39x920 with 20 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "pp = df[[\"Survived\", \"Age\", \"Fare\", \"SibSp\", \"Parch\"]].copy()\n",
    "pp[\"LogFare\"] = np.log1p(pp.pop(\"Fare\"))\n",
    "pp[\"Survived\"] = pp[\"Survived\"].map({0: \"No\", 1: \"Yes\"})\n",
    "g = sns.pairplot(pp.dropna(), hue=\"Survived\", palette={\"No\": \"#7f8c8d\", \"Yes\": \"#27ae60\"},\n",
    "                 plot_kws={\"alpha\": 0.45, \"s\": 18}, diag_kind=\"kde\", height=2.3)\n",
    "g.figure.suptitle(\"Pair Plot of Numeric Features Coloured by Survival\", y=1.02)\n",
    "plt.savefig(\"figures/cs3_07_pairplot.png\", dpi=130, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a5f0643a",
   "metadata": {},
   "source": [
    "**Interpretation:** no single pair of numeric features separates survivors cleanly — the green and grey clouds\n",
    "overlap — which is why the categorical features (sex, class) carry most of the signal. The clearest pattern is\n",
    "along `LogFare`: survivors shift to the right. `SibSp` and `Parch` are positively related to each other (families\n",
    "travel together) and very large families (SibSp ≥ 4) are almost all non-survivors."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7fbd81fc",
   "metadata": {},
   "source": [
    "#### Correlation heatmap\n",
    "**Purpose:** quantify linear relationships between numeric/ordinal features and the target, and check for\n",
    "multicollinearity before fitting a linear model."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "85a2b5c8",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:03.238773Z",
     "iopub.status.busy": "2026-10-25T17:03:03.238480Z",
     "iopub.status.idle": "2026-10-25T17:03:04.046046Z",
     "shell.execute_reply": "2026-10-25T17:03:04.044650Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 750x600 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/plain": [
       "Sex_male   -0.543351\n",
       "Pclass     -0.338481\n",
       "Age        -0.077221\n",
       "SibSp      -0.035322\n",
       "Parch       0.081629\n",
       "Fare        0.257307\n",
       "HasCabin    0.316912\n",
       "Name: Survived, dtype: float64"
      ]
     },
     "execution_count": 14,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "corr_df = df[[\"Survived\", \"Pclass\", \"Age\", \"SibSp\", \"Parch\", \"Fare\"]].copy()\n",
    "corr_df[\"Sex_male\"] = (df[\"Sex\"] == \"male\").astype(int)\n",
    "corr_df[\"HasCabin\"] = df[\"Cabin\"].notna().astype(int)\n",
    "corr = corr_df.corr()\n",
    "fig, ax = plt.subplots(figsize=(7.5, 6))\n",
    "sns.heatmap(corr, annot=True, fmt=\".2f\", cmap=\"coolwarm\", center=0, square=True,\n",
    "            cbar_kws={\"label\": \"Pearson correlation\"}, ax=ax)\n",
    "ax.set_title(\"Correlation Matrix of Features and Survival\")\n",
    "savefig(\"cs3_08_correlation\")\n",
    "corr[\"Survived\"].drop(\"Survived\").sort_values()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a5532b33",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- Strongest correlates of survival: `Sex_male` (−0.54), `Pclass` (−0.34), `HasCabin` (+0.32), `Fare` (+0.26).\n",
    "- `Age` has almost no *linear* correlation with survival (−0.08) — but note it is correlated with `Pclass`\n",
    "  (−0.37: older passengers more often in 1st class). This confounding matters and is revisited in Section 5.\n",
    "- **Multicollinearity check:** the strongest predictor–predictor correlation is `Pclass`–`HasCabin` (−0.73), followed by\n",
    "  `Pclass`–`Fare` (−0.55). Class, cabin and fare therefore carry overlapping information. This does not hurt prediction,\n",
    "  but it makes their *individual* coefficients less certain — Section 5b shows exactly this effect."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1b8a35ad",
   "metadata": {},
   "source": [
    "### 1d. Statistical tests — are the patterns real or chance?\n",
    "**Purpose:** the charts show differences; hypothesis tests check whether they are larger than random variation.\n",
    "- **Chi-square test of independence** for categorical features vs survival (H₀: feature and survival are independent).\n",
    "- **Mann–Whitney U test** for numeric features (H₀: same distribution for survivors and non-survivors). It is used\n",
    "  instead of a t-test because `Fare` and `Age` are not normally distributed.\n",
    "\n",
    "Significance level α = 0.05."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "93f5a21e",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.049052Z",
     "iopub.status.busy": "2026-10-25T17:03:04.048781Z",
     "iopub.status.idle": "2026-10-25T17:03:04.108406Z",
     "shell.execute_reply": "2026-10-25T17:03:04.107061Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Feature</th>\n",
       "      <th>Test</th>\n",
       "      <th>Statistic</th>\n",
       "      <th>dof</th>\n",
       "      <th>p-value</th>\n",
       "      <th>Effect size (Cramér's V)</th>\n",
       "      <th>Significant (α=0.05)</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Sex</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>260.72</td>\n",
       "      <td>1</td>\n",
       "      <td>1.20e-58</td>\n",
       "      <td>0.541</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Pclass</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>102.89</td>\n",
       "      <td>2</td>\n",
       "      <td>4.55e-23</td>\n",
       "      <td>0.34</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Embarked</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>26.49</td>\n",
       "      <td>2</td>\n",
       "      <td>1.77e-06</td>\n",
       "      <td>0.173</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>HasCabin</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>87.94</td>\n",
       "      <td>1</td>\n",
       "      <td>6.74e-21</td>\n",
       "      <td>0.314</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>Age</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>57682.00</td>\n",
       "      <td>-</td>\n",
       "      <td>1.60e-01</td>\n",
       "      <td>-</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>Fare</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>129951.50</td>\n",
       "      <td>-</td>\n",
       "      <td>4.55e-22</td>\n",
       "      <td>-</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "    Feature            Test  Statistic dof   p-value Effect size (Cramér's V)  \\\n",
       "0       Sex      Chi-square     260.72   1  1.20e-58                    0.541   \n",
       "1    Pclass      Chi-square     102.89   2  4.55e-23                     0.34   \n",
       "2  Embarked      Chi-square      26.49   2  1.77e-06                    0.173   \n",
       "3  HasCabin      Chi-square      87.94   1  6.74e-21                    0.314   \n",
       "4       Age  Mann-Whitney U   57682.00   -  1.60e-01                        -   \n",
       "5      Fare  Mann-Whitney U  129951.50   -  4.55e-22                        -   \n",
       "\n",
       "  Significant (α=0.05)  \n",
       "0                  Yes  \n",
       "1                  Yes  \n",
       "2                  Yes  \n",
       "3                  Yes  \n",
       "4                   No  \n",
       "5                  Yes  "
      ]
     },
     "execution_count": 15,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df[\"HasCabin\"] = df[\"Cabin\"].notna().astype(int)\n",
    "rows = []\n",
    "for col in [\"Sex\", \"Pclass\", \"Embarked\", \"HasCabin\"]:\n",
    "    table = pd.crosstab(df[col], df[\"Survived\"])\n",
    "    chi2, p, dof, _ = stats.chi2_contingency(table)\n",
    "    n = table.values.sum()\n",
    "    cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))\n",
    "    rows.append({\"Feature\": col, \"Test\": \"Chi-square\", \"Statistic\": round(chi2, 2), \"dof\": dof,\n",
    "                 \"p-value\": f\"{p:.2e}\", \"Effect size (Cramér's V)\": round(cramers_v, 3),\n",
    "                 \"Significant (α=0.05)\": \"Yes\" if p < 0.05 else \"No\"})\n",
    "for col in [\"Age\", \"Fare\"]:\n",
    "    a = df.loc[df.Survived == 1, col].dropna()\n",
    "    b = df.loc[df.Survived == 0, col].dropna()\n",
    "    u, p = stats.mannwhitneyu(a, b)\n",
    "    rows.append({\"Feature\": col, \"Test\": \"Mann-Whitney U\", \"Statistic\": round(u, 1), \"dof\": \"-\",\n",
    "                 \"p-value\": f\"{p:.2e}\", \"Effect size (Cramér's V)\": \"-\",\n",
    "                 \"Significant (α=0.05)\": \"Yes\" if p < 0.05 else \"No\"})\n",
    "stat_tests = pd.DataFrame(rows)\n",
    "stat_tests"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c1460add",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- `Sex` (χ² = 260.7, Cramér's V ≈ 0.54, a large effect), `Pclass` (χ² = 102.9), `HasCabin` (χ² = 87.9) and `Embarked`\n",
    "  (χ² = 26.5) are all significantly associated with survival (p ≪ 0.05).\n",
    "- `Fare` differs significantly between groups (median £26.0 for survivors vs £10.5 for non-survivors, p ≈ 4.6 × 10⁻²²).\n",
    "- **`Age` is *not* significant on its own (p = 0.16)** — survivors and non-survivors both have a median age of 28.\n",
    "  This does not mean age is irrelevant; Section 5 shows that once class is held constant, age becomes a significant\n",
    "  negative factor."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a3c6daf9",
   "metadata": {},
   "source": [
    "## 2. Data Preprocessing\n",
    "### 2a. Feature engineering and selection\n",
    "**Purpose:** turn raw columns into model-ready inputs.\n",
    "\n",
    "| Raw column | Treatment | Reason |\n",
    "|---|---|---|\n",
    "| `PassengerId`, `Name`, `Ticket` | dropped | identifiers with no generalisable signal |\n",
    "| `Cabin` | → `HasCabin` (1/0) | 77% missing; *presence* of a cabin record is informative |\n",
    "| `Age` | median imputation + `AgeMissing` flag | keeps 177 rows; flag lets the model learn if \"unknown age\" itself matters |\n",
    "| `Fare` | → `LogFare = log(1 + Fare)` | reduces skewness from 4.8 to 0.39 so extreme fares don't dominate |\n",
    "| `Embarked` | most-frequent imputation, one-hot | only 2 missing |\n",
    "| `Pclass`, `Sex` | one-hot encoded (first level dropped) | categorical; dropping one level avoids perfect collinearity |\n",
    "| numeric columns | standardised (mean 0, SD 1) | puts coefficients on a common per-SD scale and helps the solver converge |"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "a1ddd249",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.111130Z",
     "iopub.status.busy": "2026-10-25T17:03:04.110868Z",
     "iopub.status.idle": "2026-10-25T17:03:04.128551Z",
     "shell.execute_reply": "2026-10-25T17:03:04.127385Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Age</th>\n",
       "      <th>SibSp</th>\n",
       "      <th>Parch</th>\n",
       "      <th>LogFare</th>\n",
       "      <th>Pclass</th>\n",
       "      <th>Sex</th>\n",
       "      <th>Embarked</th>\n",
       "      <th>HasCabin</th>\n",
       "      <th>AgeMissing</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>22.0</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "      <td>2.110213</td>\n",
       "      <td>3</td>\n",
       "      <td>male</td>\n",
       "      <td>S</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>38.0</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "      <td>4.280593</td>\n",
       "      <td>1</td>\n",
       "      <td>female</td>\n",
       "      <td>C</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>26.0</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>2.188856</td>\n",
       "      <td>3</td>\n",
       "      <td>female</td>\n",
       "      <td>S</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>35.0</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "      <td>3.990834</td>\n",
       "      <td>1</td>\n",
       "      <td>female</td>\n",
       "      <td>S</td>\n",
       "      <td>1</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>35.0</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>2.202765</td>\n",
       "      <td>3</td>\n",
       "      <td>male</td>\n",
       "      <td>S</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "    Age  SibSp  Parch   LogFare  Pclass     Sex Embarked  HasCabin  AgeMissing\n",
       "0  22.0      1      0  2.110213       3    male        S         0           0\n",
       "1  38.0      1      0  4.280593       1  female        C         1           0\n",
       "2  26.0      0      0  2.188856       3  female        S         0           0\n",
       "3  35.0      1      0  3.990834       1  female        S         1           0\n",
       "4  35.0      0      0  2.202765       3    male        S         0           0"
      ]
     },
     "execution_count": 16,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "data = df.copy()\n",
    "data[\"HasCabin\"] = data[\"Cabin\"].notna().astype(int)\n",
    "data[\"AgeMissing\"] = data[\"Age\"].isna().astype(int)\n",
    "data[\"LogFare\"] = np.log1p(data[\"Fare\"])\n",
    "\n",
    "NUM_FEATURES = [\"Age\", \"SibSp\", \"Parch\", \"LogFare\"]\n",
    "CAT_FEATURES = [\"Pclass\", \"Sex\", \"Embarked\"]\n",
    "BIN_FEATURES = [\"HasCabin\", \"AgeMissing\"]\n",
    "FEATURES = NUM_FEATURES + CAT_FEATURES + BIN_FEATURES\n",
    "\n",
    "X = data[FEATURES]\n",
    "y = data[\"Survived\"]\n",
    "X.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2b89ba52",
   "metadata": {},
   "source": [
    "### 2b. Train / test split\n",
    "**Purpose:** hold out 20% of passengers the model never sees during training, to get an honest estimate of\n",
    "performance on new data. `stratify=y` keeps the 62/38 class ratio identical in both sets."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "id": "192696bf",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.131292Z",
     "iopub.status.busy": "2026-10-25T17:03:04.131050Z",
     "iopub.status.idle": "2026-10-25T17:03:04.170459Z",
     "shell.execute_reply": "2026-10-25T17:03:04.169205Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Training set: 712 rows | survival rate 38.3%\n",
      "Test set    : 179 rows | survival rate 38.5%\n"
     ]
    }
   ],
   "source": [
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)\n",
    "print(f\"Training set: {X_train.shape[0]} rows | survival rate {y_train.mean():.1%}\")\n",
    "print(f\"Test set    : {X_test.shape[0]} rows | survival rate {y_test.mean():.1%}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7252be31",
   "metadata": {},
   "source": [
    "### 2c. Preprocessing pipeline (missing values + encoding + scaling)\n",
    "**Purpose:** wrap imputation, encoding and scaling in a scikit-learn `ColumnTransformer` inside a `Pipeline`.\n",
    "This matters for correctness: the medians, most-frequent values and scaling statistics are learned **only from\n",
    "the training data** and then applied to the test data, so no information leaks from the test set into training.\n",
    "The same fitted pipeline is later saved and reused by the Streamlit app, so the app applies the identical transformations."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "156dcf5d",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.173292Z",
     "iopub.status.busy": "2026-10-25T17:03:04.173060Z",
     "iopub.status.idle": "2026-10-25T17:03:04.214301Z",
     "shell.execute_reply": "2026-10-25T17:03:04.213503Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Transformed shape: (712, 11)\n",
      "Missing values after preprocessing: 0\n",
      "Model input columns: ['num__Age', 'num__SibSp', 'num__Parch', 'num__LogFare', 'cat__Pclass_2', 'cat__Pclass_3', 'cat__Sex_male', 'cat__Embarked_Q', 'cat__Embarked_S', 'bin__HasCabin', 'bin__AgeMissing']\n"
     ]
    }
   ],
   "source": [
    "numeric_pipe = Pipeline([\n",
    "    (\"impute\", SimpleImputer(strategy=\"median\")),\n",
    "    (\"scale\", StandardScaler()),\n",
    "])\n",
    "categorical_pipe = Pipeline([\n",
    "    (\"impute\", SimpleImputer(strategy=\"most_frequent\")),\n",
    "    (\"onehot\", OneHotEncoder(drop=\"first\", handle_unknown=\"ignore\")),\n",
    "])\n",
    "preprocessor = ColumnTransformer([\n",
    "    (\"num\", numeric_pipe, NUM_FEATURES),\n",
    "    (\"cat\", categorical_pipe, CAT_FEATURES),\n",
    "    (\"bin\", \"passthrough\", BIN_FEATURES),\n",
    "])\n",
    "\n",
    "# Check the transformed training data: no missing values should remain.\n",
    "Xt = preprocessor.fit_transform(X_train)\n",
    "print(\"Transformed shape:\", Xt.shape)\n",
    "print(\"Missing values after preprocessing:\", int(np.isnan(Xt).sum()))\n",
    "print(\"Model input columns:\", list(preprocessor.get_feature_names_out()))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b584d142",
   "metadata": {},
   "source": [
    "**Observation:** the 9 raw features become 11 model inputs (one-hot encoding expands `Pclass` and `Embarked` into two\n",
    "columns each; `Sex` becomes one), and no missing values remain. Baseline (dropped) categories are\n",
    "`Pclass = 1`, `Sex = female`, `Embarked = C`, so every categorical coefficient is read *relative to* those."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "240a2224",
   "metadata": {},
   "source": [
    "## 3. Model Building and Training\n",
    "**Purpose:** fit a logistic regression on the training data.\n",
    "\n",
    "Logistic regression computes\n",
    "  **P(survive) = 1 / (1 + e^−(β₀ + β₁x₁ + … + βₖxₖ))**\n",
    "— a weighted sum of features squashed by the sigmoid into a probability between 0 and 1. Training finds the weights β that\n",
    "maximise the likelihood of the observed outcomes. scikit-learn's default L2 regularisation (`C = 1.0`) is kept: it\n",
    "slightly shrinks coefficients to reduce over-fitting on this small dataset. A passenger is classified as a\n",
    "survivor when P(survive) ≥ 0.5."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "cc43951c",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.219122Z",
     "iopub.status.busy": "2026-10-25T17:03:04.218860Z",
     "iopub.status.idle": "2026-10-25T17:03:04.282395Z",
     "shell.execute_reply": "2026-10-25T17:03:04.281190Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Training accuracy: 0.808\n"
     ]
    }
   ],
   "source": [
    "model = Pipeline([\n",
    "    (\"preprocess\", preprocessor),\n",
    "    (\"logreg\", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),\n",
    "])\n",
    "model.fit(X_train, y_train)\n",
    "\n",
    "train_acc = accuracy_score(y_train, model.predict(X_train))\n",
    "print(f\"Training accuracy: {train_acc:.3f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6251a735",
   "metadata": {},
   "source": [
    "### 3b. Cross-validation on the training set\n",
    "**Purpose:** a single test split can be lucky or unlucky. 5-fold stratified cross-validation trains and scores the\n",
    "model five times on different slices of the training data, giving a mean and spread for each metric."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "c35be8e7",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.284804Z",
     "iopub.status.busy": "2026-10-25T17:03:04.284597Z",
     "iopub.status.idle": "2026-10-25T17:03:04.533901Z",
     "shell.execute_reply": "2026-10-25T17:03:04.532828Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>mean</th>\n",
       "      <th>std</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>accuracy</th>\n",
       "      <td>0.799</td>\n",
       "      <td>0.013</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>precision</th>\n",
       "      <td>0.753</td>\n",
       "      <td>0.037</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>recall</th>\n",
       "      <td>0.714</td>\n",
       "      <td>0.038</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>f1</th>\n",
       "      <td>0.732</td>\n",
       "      <td>0.015</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>roc_auc</th>\n",
       "      <td>0.864</td>\n",
       "      <td>0.020</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "            mean    std\n",
       "accuracy   0.799  0.013\n",
       "precision  0.753  0.037\n",
       "recall     0.714  0.038\n",
       "f1         0.732  0.015\n",
       "roc_auc    0.864  0.020"
      ]
     },
     "execution_count": 20,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n",
    "cv_res = cross_validate(model, X_train, y_train, cv=cv,\n",
    "                        scoring=[\"accuracy\", \"precision\", \"recall\", \"f1\", \"roc_auc\"])\n",
    "cv_table = pd.DataFrame({m: [cv_res[f\"test_{m}\"].mean(), cv_res[f\"test_{m}\"].std()]\n",
    "                         for m in [\"accuracy\", \"precision\", \"recall\", \"f1\", \"roc_auc\"]},\n",
    "                        index=[\"mean\", \"std\"]).T.round(3)\n",
    "cv_table"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fc7a3216",
   "metadata": {},
   "source": [
    "## 4. Model Evaluation on the Test Set\n",
    "**Purpose:** score the trained model on the 179 held-out passengers.\n",
    "\n",
    "- **Accuracy** — share of all predictions that are correct.\n",
    "- **Precision** — of the passengers predicted to survive, the share who actually survived.\n",
    "- **Recall** — of the passengers who actually survived, the share the model found.\n",
    "- **F1-score** — harmonic mean of precision and recall (high only if both are high).\n",
    "- **ROC-AUC** — probability that the model ranks a random survivor above a random non-survivor; 0.5 = coin flip, 1.0 = perfect. Unlike the others it does not depend on the 0.5 threshold."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 21,
   "id": "66eb3c06",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.535986Z",
     "iopub.status.busy": "2026-10-25T17:03:04.535807Z",
     "iopub.status.idle": "2026-10-25T17:03:04.567799Z",
     "shell.execute_reply": "2026-10-25T17:03:04.566511Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Baseline accuracy (always predict 'did not survive'): 0.615\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Metric</th>\n",
       "      <th>Test score</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Accuracy</td>\n",
       "      <td>0.793</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Precision</td>\n",
       "      <td>0.767</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Recall</td>\n",
       "      <td>0.667</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>F1-score</td>\n",
       "      <td>0.713</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>ROC-AUC</td>\n",
       "      <td>0.833</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "      Metric  Test score\n",
       "0   Accuracy       0.793\n",
       "1  Precision       0.767\n",
       "2     Recall       0.667\n",
       "3   F1-score       0.713\n",
       "4    ROC-AUC       0.833"
      ]
     },
     "execution_count": 21,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "y_pred = model.predict(X_test)\n",
    "y_prob = model.predict_proba(X_test)[:, 1]\n",
    "\n",
    "metrics = pd.DataFrame({\n",
    "    \"Metric\": [\"Accuracy\", \"Precision\", \"Recall\", \"F1-score\", \"ROC-AUC\"],\n",
    "    \"Test score\": [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),\n",
    "                   recall_score(y_test, y_pred), f1_score(y_test, y_pred), roc_auc_score(y_test, y_prob)],\n",
    "}).round(3)\n",
    "baseline_acc = (y_test == 0).mean()\n",
    "print(f\"Baseline accuracy (always predict 'did not survive'): {baseline_acc:.3f}\")\n",
    "metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 22,
   "id": "afe0f32a",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.569738Z",
     "iopub.status.busy": "2026-10-25T17:03:04.569555Z",
     "iopub.status.idle": "2026-10-25T17:03:04.580752Z",
     "shell.execute_reply": "2026-10-25T17:03:04.579869Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "                 precision    recall  f1-score   support\n",
      "\n",
      "Did not survive      0.807     0.873     0.838       110\n",
      "       Survived      0.767     0.667     0.713        69\n",
      "\n",
      "       accuracy                          0.793       179\n",
      "      macro avg      0.787     0.770     0.776       179\n",
      "   weighted avg      0.791     0.793     0.790       179\n",
      "\n"
     ]
    }
   ],
   "source": [
    "print(classification_report(y_test, y_pred, target_names=[\"Did not survive\", \"Survived\"], digits=3))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c6298697",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Accuracy 0.793** versus a 0.615 baseline — the model correctly classifies about 8 in 10 unseen passengers, about 18\n",
    "  percentage points better than always guessing \"died\".\n",
    "- **Precision 0.767:** when the model says \"survived\", it is right about 77% of the time.\n",
    "- **Recall 0.667:** it finds two-thirds of actual survivors; one-third are missed (mostly passengers whose profile — e.g.\n",
    "  an adult 3rd-class man — normally implies death but who survived anyway).\n",
    "- **F1 0.713** balances the two. The \"Did not survive\" class scores higher (F1 0.838) because it is the larger, easier class.\n",
    "- **ROC-AUC 0.833:** in 83% of survivor/non-survivor pairs the model gives the survivor the higher probability — good\n",
    "  discrimination for a model with only 9 raw inputs.\n",
    "- **Generalisation:** training accuracy (0.808) is close to test accuracy (0.793), and cross-validated scores\n",
    "  (accuracy 0.799 ± 0.013, ROC-AUC 0.864 ± 0.020) agree with the test scores, so the model is not over-fitting.\n",
    "  The test AUC is slightly below the CV mean — normal variation for a 179-passenger test set."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7f6dd3f8",
   "metadata": {},
   "source": [
    "### 4b. Confusion matrix\n",
    "**Purpose:** see exactly which kind of mistakes the model makes."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 23,
   "id": "2bb912dc",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.585030Z",
     "iopub.status.busy": "2026-10-25T17:03:04.584778Z",
     "iopub.status.idle": "2026-10-25T17:03:04.799127Z",
     "shell.execute_reply": "2026-10-25T17:03:04.798168Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 550x450 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "TN=96  FP=14  FN=23  TP=46\n"
     ]
    }
   ],
   "source": [
    "cm = confusion_matrix(y_test, y_pred)\n",
    "fig, ax = plt.subplots(figsize=(5.5, 4.5))\n",
    "ConfusionMatrixDisplay(cm, display_labels=[\"Did not survive\", \"Survived\"]).plot(ax=ax, cmap=\"Blues\", colorbar=False)\n",
    "ax.grid(False)\n",
    "ax.set_title(\"Confusion Matrix — Logistic Regression (Test Set)\")\n",
    "ax.set_xlabel(\"Predicted outcome\")\n",
    "ax.set_ylabel(\"Actual outcome\")\n",
    "savefig(\"cs3_09_confusion_matrix\")\n",
    "tn, fp, fn, tp = cm.ravel()\n",
    "print(f\"TN={tn}  FP={fp}  FN={fn}  TP={tp}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e16f4d80",
   "metadata": {},
   "source": [
    "**Interpretation:** of 110 non-survivors, 96 are correctly identified and 14 are wrongly predicted to survive\n",
    "(false positives). Of 69 survivors, 46 are found and 23 are missed (false negatives). The model errs more on the\n",
    "side of predicting death — expected, since death is the majority class and the default 0.5 threshold favours it.\n",
    "If finding survivors mattered more (higher recall), the threshold could be lowered, trading some precision."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "98df8d1c",
   "metadata": {},
   "source": [
    "### 4c. ROC curve\n",
    "**Purpose:** show the trade-off between true-positive rate and false-positive rate at every possible threshold."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 24,
   "id": "1805cb6b",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:04.801719Z",
     "iopub.status.busy": "2026-10-25T17:03:04.801535Z",
     "iopub.status.idle": "2026-10-25T17:03:05.219339Z",
     "shell.execute_reply": "2026-10-25T17:03:05.217542Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 600x500 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fpr, tpr, thr = roc_curve(y_test, y_prob)\n",
    "auc = roc_auc_score(y_test, y_prob)\n",
    "i05 = np.argmin(np.abs(thr - 0.5))\n",
    "fig, ax = plt.subplots(figsize=(6, 5))\n",
    "ax.plot(fpr, tpr, color=\"#2c7fb8\", lw=2.2, label=f\"Logistic regression (AUC = {auc:.3f})\")\n",
    "ax.plot([0, 1], [0, 1], \"--\", color=\"grey\", label=\"Random guess (AUC = 0.5)\")\n",
    "ax.scatter(fpr[i05], tpr[i05], color=\"red\", zorder=5, label=\"Threshold = 0.5\")\n",
    "ax.set_title(\"ROC Curve — Titanic Survival (Test Set)\")\n",
    "ax.set_xlabel(\"False positive rate (1 − specificity)\")\n",
    "ax.set_ylabel(\"True positive rate (recall)\")\n",
    "ax.legend(loc=\"lower right\")\n",
    "savefig(\"cs3_10_roc_curve\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0fbca8c0",
   "metadata": {},
   "source": [
    "**Interpretation:** the curve bows well above the diagonal (AUC 0.833). The red point is the default 0.5 threshold:\n",
    "≈ 67% of survivors caught at a ≈ 13% false-positive rate (14 of 110 non-survivors). Moving left along the curve (higher threshold) makes\n",
    "\"survived\" predictions more reliable; moving right catches more survivors at the cost of more false alarms."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c9ec8395",
   "metadata": {},
   "source": [
    "### 4d. How the threshold changes precision and recall\n",
    "**Purpose:** make the threshold trade-off concrete, since 0.5 is a default, not a law."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 25,
   "id": "8ea33fae",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:05.221984Z",
     "iopub.status.busy": "2026-10-25T17:03:05.221750Z",
     "iopub.status.idle": "2026-10-25T17:03:06.162021Z",
     "shell.execute_reply": "2026-10-25T17:03:06.160407Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 700x420 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Threshold maximising F1 on the test set: 0.44 (F1 = 0.748)\n"
     ]
    }
   ],
   "source": [
    "ths = np.linspace(0.1, 0.9, 81)\n",
    "prec = [precision_score(y_test, (y_prob >= t).astype(int), zero_division=0) for t in ths]\n",
    "rec = [recall_score(y_test, (y_prob >= t).astype(int)) for t in ths]\n",
    "f1s = [f1_score(y_test, (y_prob >= t).astype(int)) for t in ths]\n",
    "fig, ax = plt.subplots(figsize=(7, 4.2))\n",
    "ax.plot(ths, prec, label=\"Precision\")\n",
    "ax.plot(ths, rec, label=\"Recall\")\n",
    "ax.plot(ths, f1s, label=\"F1-score\", lw=2.4)\n",
    "ax.axvline(0.5, ls=\"--\", c=\"grey\", lw=1, label=\"Default threshold 0.5\")\n",
    "ax.set_title(\"Precision, Recall and F1 vs Classification Threshold\")\n",
    "ax.set_xlabel(\"Probability threshold for predicting 'Survived'\")\n",
    "ax.set_ylabel(\"Score\")\n",
    "ax.legend()\n",
    "savefig(\"cs3_11_threshold\")\n",
    "best_t = ths[int(np.argmax(f1s))]\n",
    "print(f\"Threshold maximising F1 on the test set: {best_t:.2f} (F1 = {max(f1s):.3f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a1e36d29",
   "metadata": {},
   "source": [
    "**Interpretation:** lowering the threshold raises recall and lowers precision; raising it does the opposite. F1 stays\n",
    "within about 0.71–0.75 for any threshold between 0.30 and 0.60 (peak 0.748 at 0.44), so the model is not fragile to the\n",
    "exact cut-off and the default 0.5 is a reasonable choice. The threshold is **not** re-tuned\n",
    "on the test set — doing so would leak test information and inflate the reported score; this chart is diagnostic only."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8132556b",
   "metadata": {},
   "source": [
    "## 5. Interpretation of the Model\n",
    "### 5a. Coefficients and odds ratios\n",
    "**Purpose:** read what the model learned. A coefficient β is the change in **log-odds** of survival for a one-unit\n",
    "increase in that input (one standard deviation for numeric features; switching from the baseline category for\n",
    "encoded ones). **Odds ratio = e^β**: values above 1 raise the odds of survival, below 1 lower them."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 26,
   "id": "60a5c7e6",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:06.164597Z",
     "iopub.status.busy": "2026-10-25T17:03:06.164310Z",
     "iopub.status.idle": "2026-10-25T17:03:06.180048Z",
     "shell.execute_reply": "2026-10-25T17:03:06.178913Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Intercept: 1.399\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Feature</th>\n",
       "      <th>Coefficient (log-odds)</th>\n",
       "      <th>Odds ratio</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Sex_male</td>\n",
       "      <td>-2.486</td>\n",
       "      <td>0.083</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Pclass_3</td>\n",
       "      <td>-0.749</td>\n",
       "      <td>0.473</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>Age</td>\n",
       "      <td>-0.488</td>\n",
       "      <td>0.614</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>Embarked_S</td>\n",
       "      <td>-0.404</td>\n",
       "      <td>0.668</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>SibSp</td>\n",
       "      <td>-0.356</td>\n",
       "      <td>0.700</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>AgeMissing</td>\n",
       "      <td>-0.221</td>\n",
       "      <td>0.802</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>Parch</td>\n",
       "      <td>-0.174</td>\n",
       "      <td>0.841</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>Pclass_2</td>\n",
       "      <td>0.147</td>\n",
       "      <td>1.158</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>Embarked_Q</td>\n",
       "      <td>0.396</td>\n",
       "      <td>1.487</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>LogFare</td>\n",
       "      <td>0.453</td>\n",
       "      <td>1.573</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>HasCabin</td>\n",
       "      <td>1.072</td>\n",
       "      <td>2.922</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "       Feature  Coefficient (log-odds)  Odds ratio\n",
       "0     Sex_male                  -2.486       0.083\n",
       "1     Pclass_3                  -0.749       0.473\n",
       "2          Age                  -0.488       0.614\n",
       "3   Embarked_S                  -0.404       0.668\n",
       "4        SibSp                  -0.356       0.700\n",
       "5   AgeMissing                  -0.221       0.802\n",
       "6        Parch                  -0.174       0.841\n",
       "7     Pclass_2                   0.147       1.158\n",
       "8   Embarked_Q                   0.396       1.487\n",
       "9      LogFare                   0.453       1.573\n",
       "10    HasCabin                   1.072       2.922"
      ]
     },
     "execution_count": 26,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "feat_names = [n.split(\"__\", 1)[1] for n in model.named_steps[\"preprocess\"].get_feature_names_out()]\n",
    "coefs = model.named_steps[\"logreg\"].coef_[0]\n",
    "coef_table = pd.DataFrame({\"Feature\": feat_names, \"Coefficient (log-odds)\": coefs,\n",
    "                           \"Odds ratio\": np.exp(coefs)}).sort_values(\"Coefficient (log-odds)\")\n",
    "print(f\"Intercept: {model.named_steps['logreg'].intercept_[0]:.3f}\")\n",
    "coef_table.round(3).reset_index(drop=True)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 27,
   "id": "1b474481",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:06.182527Z",
     "iopub.status.busy": "2026-10-25T17:03:06.182234Z",
     "iopub.status.idle": "2026-10-25T17:03:06.585383Z",
     "shell.execute_reply": "2026-10-25T17:03:06.584099Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 800x500 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "ct = coef_table.sort_values(\"Coefficient (log-odds)\")\n",
    "colors = [\"#c0392b\" if c < 0 else \"#27ae60\" for c in ct[\"Coefficient (log-odds)\"]]\n",
    "fig, ax = plt.subplots(figsize=(8, 5))\n",
    "ax.barh(ct[\"Feature\"], ct[\"Coefficient (log-odds)\"], color=colors)\n",
    "ax.axvline(0, color=\"black\", lw=0.8)\n",
    "for y_, (c, o) in enumerate(zip(ct[\"Coefficient (log-odds)\"], ct[\"Odds ratio\"])):\n",
    "    ax.text(c + (0.05 if c >= 0 else -0.05), y_, f\"OR={o:.2f}\", va=\"center\",\n",
    "            ha=\"left\" if c >= 0 else \"right\", fontsize=9)\n",
    "ax.set_title(\"Logistic Regression Coefficients (with Odds Ratios)\")\n",
    "ax.set_xlabel(\"Coefficient (change in log-odds of survival)\")\n",
    "ax.set_ylabel(\"Model feature\")\n",
    "ax.set_xlim(-3.3, 1.8)\n",
    "savefig(\"cs3_12_coefficients\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a3c6a860",
   "metadata": {},
   "source": [
    "**Interpretation (holding all other features constant)**\n",
    "- **Sex_male (β ≈ −2.49, OR ≈ 0.08):** being male multiplies the odds of survival by about 0.08 — roughly **12 times\n",
    "  lower odds** than a comparable woman. By far the dominant factor.\n",
    "- **HasCabin (OR ≈ 2.9):** passengers with a recorded cabin had about 2.9× the odds of survival — a proxy for deck\n",
    "  location and status beyond what `Pclass` captures.\n",
    "- **Pclass_3 (OR ≈ 0.47):** 3rd class roughly halves the odds relative to 1st class, on top of the cabin and fare effects.\n",
    "  **Pclass_2** is close to 1st class once cabin and fare are accounted for. (5b shows these class estimates are individually\n",
    "  uncertain because class overlaps with cabin and fare.)\n",
    "- **Age (OR ≈ 0.61 per SD ≈ 13 years):** each ~13 extra years cuts the odds of survival by ~39% — older passengers fared\n",
    "  worse *at the same class and sex*.\n",
    "- **SibSp (OR ≈ 0.70):** more siblings/spouses aboard lowered survival — large families were harder to evacuate together.\n",
    "- **LogFare (OR ≈ 1.6):** higher fare raised the odds, even within class.\n",
    "- **Embarked:** relative to Cherbourg, Southampton lowers and Queenstown slightly raises the odds; these effects are small and,\n",
    "  as the significance test below shows, not statistically reliable."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "baebd23b",
   "metadata": {},
   "source": [
    "### 5b. Statistical significance of each feature\n",
    "**Purpose:** scikit-learn does not report p-values. Re-fitting the same specification with `statsmodels` (unregularised\n",
    "maximum likelihood, on the same preprocessed training data) gives standard errors, p-values and 95% confidence\n",
    "intervals, so we can say which effects are statistically reliable rather than noise."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 28,
   "id": "ed3f9827",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:06.587705Z",
     "iopub.status.busy": "2026-10-25T17:03:06.587523Z",
     "iopub.status.idle": "2026-10-25T17:03:06.631712Z",
     "shell.execute_reply": "2026-10-25T17:03:06.630512Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "McFadden pseudo R²: 0.364 | LLR p-value: 2.53e-67\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Coef</th>\n",
       "      <th>Std err</th>\n",
       "      <th>z</th>\n",
       "      <th>p-value</th>\n",
       "      <th>Odds ratio</th>\n",
       "      <th>OR 95% CI low</th>\n",
       "      <th>OR 95% CI high</th>\n",
       "      <th>Significant (p&lt;0.05)</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Sex_male</th>\n",
       "      <td>-2.6233</td>\n",
       "      <td>0.2274</td>\n",
       "      <td>-11.5339</td>\n",
       "      <td>0.0000</td>\n",
       "      <td>0.0726</td>\n",
       "      <td>0.0465</td>\n",
       "      <td>0.1133</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Age</th>\n",
       "      <td>-0.5096</td>\n",
       "      <td>0.1153</td>\n",
       "      <td>-4.4200</td>\n",
       "      <td>0.0000</td>\n",
       "      <td>0.6007</td>\n",
       "      <td>0.4792</td>\n",
       "      <td>0.7530</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>HasCabin</th>\n",
       "      <td>1.1582</td>\n",
       "      <td>0.3770</td>\n",
       "      <td>3.0719</td>\n",
       "      <td>0.0021</td>\n",
       "      <td>3.1842</td>\n",
       "      <td>1.5208</td>\n",
       "      <td>6.6669</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>SibSp</th>\n",
       "      <td>-0.3660</td>\n",
       "      <td>0.1425</td>\n",
       "      <td>-2.5682</td>\n",
       "      <td>0.0102</td>\n",
       "      <td>0.6935</td>\n",
       "      <td>0.5245</td>\n",
       "      <td>0.9170</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>LogFare</th>\n",
       "      <td>0.4346</td>\n",
       "      <td>0.1873</td>\n",
       "      <td>2.3198</td>\n",
       "      <td>0.0203</td>\n",
       "      <td>1.5443</td>\n",
       "      <td>1.0697</td>\n",
       "      <td>2.2294</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Parch</th>\n",
       "      <td>-0.1858</td>\n",
       "      <td>0.1135</td>\n",
       "      <td>-1.6366</td>\n",
       "      <td>0.1017</td>\n",
       "      <td>0.8305</td>\n",
       "      <td>0.6648</td>\n",
       "      <td>1.0374</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Pclass_3</th>\n",
       "      <td>-0.8187</td>\n",
       "      <td>0.5128</td>\n",
       "      <td>-1.5965</td>\n",
       "      <td>0.1104</td>\n",
       "      <td>0.4410</td>\n",
       "      <td>0.1614</td>\n",
       "      <td>1.2049</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Embarked_S</th>\n",
       "      <td>-0.4057</td>\n",
       "      <td>0.2759</td>\n",
       "      <td>-1.4702</td>\n",
       "      <td>0.1415</td>\n",
       "      <td>0.6665</td>\n",
       "      <td>0.3881</td>\n",
       "      <td>1.1447</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Embarked_Q</th>\n",
       "      <td>0.4707</td>\n",
       "      <td>0.4598</td>\n",
       "      <td>1.0238</td>\n",
       "      <td>0.3059</td>\n",
       "      <td>1.6011</td>\n",
       "      <td>0.6502</td>\n",
       "      <td>3.9425</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>AgeMissing</th>\n",
       "      <td>-0.2483</td>\n",
       "      <td>0.3001</td>\n",
       "      <td>-0.8271</td>\n",
       "      <td>0.4082</td>\n",
       "      <td>0.7802</td>\n",
       "      <td>0.4332</td>\n",
       "      <td>1.4049</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Pclass_2</th>\n",
       "      <td>0.1326</td>\n",
       "      <td>0.4660</td>\n",
       "      <td>0.2845</td>\n",
       "      <td>0.7760</td>\n",
       "      <td>1.1418</td>\n",
       "      <td>0.4581</td>\n",
       "      <td>2.8459</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "              Coef  Std err        z  p-value  Odds ratio  OR 95% CI low  \\\n",
       "Sex_male   -2.6233   0.2274 -11.5339   0.0000      0.0726         0.0465   \n",
       "Age        -0.5096   0.1153  -4.4200   0.0000      0.6007         0.4792   \n",
       "HasCabin    1.1582   0.3770   3.0719   0.0021      3.1842         1.5208   \n",
       "SibSp      -0.3660   0.1425  -2.5682   0.0102      0.6935         0.5245   \n",
       "LogFare     0.4346   0.1873   2.3198   0.0203      1.5443         1.0697   \n",
       "Parch      -0.1858   0.1135  -1.6366   0.1017      0.8305         0.6648   \n",
       "Pclass_3   -0.8187   0.5128  -1.5965   0.1104      0.4410         0.1614   \n",
       "Embarked_S -0.4057   0.2759  -1.4702   0.1415      0.6665         0.3881   \n",
       "Embarked_Q  0.4707   0.4598   1.0238   0.3059      1.6011         0.6502   \n",
       "AgeMissing -0.2483   0.3001  -0.8271   0.4082      0.7802         0.4332   \n",
       "Pclass_2    0.1326   0.4660   0.2845   0.7760      1.1418         0.4581   \n",
       "\n",
       "            OR 95% CI high Significant (p<0.05)  \n",
       "Sex_male            0.1133                  Yes  \n",
       "Age                 0.7530                  Yes  \n",
       "HasCabin            6.6669                  Yes  \n",
       "SibSp               0.9170                  Yes  \n",
       "LogFare             2.2294                  Yes  \n",
       "Parch               1.0374                   No  \n",
       "Pclass_3            1.2049                   No  \n",
       "Embarked_S          1.1447                   No  \n",
       "Embarked_Q          3.9425                   No  \n",
       "AgeMissing          1.4049                   No  \n",
       "Pclass_2            2.8459                   No  "
      ]
     },
     "execution_count": 28,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "Xtr_t = model.named_steps[\"preprocess\"].transform(X_train)\n",
    "Xtr_sm = sm.add_constant(pd.DataFrame(Xtr_t, columns=feat_names, index=X_train.index))\n",
    "logit = sm.Logit(y_train, Xtr_sm).fit(disp=0)\n",
    "\n",
    "sig = pd.DataFrame({\n",
    "    \"Coef\": logit.params, \"Std err\": logit.bse, \"z\": logit.tvalues, \"p-value\": logit.pvalues,\n",
    "    \"Odds ratio\": np.exp(logit.params),\n",
    "    \"OR 95% CI low\": np.exp(logit.conf_int()[0]), \"OR 95% CI high\": np.exp(logit.conf_int()[1]),\n",
    "}).drop(\"const\")\n",
    "sig[\"Significant (p<0.05)\"] = np.where(sig[\"p-value\"] < 0.05, \"Yes\", \"No\")\n",
    "print(f\"McFadden pseudo R²: {logit.prsquared:.3f} | LLR p-value: {logit.llr_pvalue:.2e}\")\n",
    "sig.sort_values(\"p-value\").round(4)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1b04ed5f",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Individually significant (p < 0.05): `Sex_male` (OR 0.07), `Age` (OR 0.60 per SD), `HasCabin` (OR 3.18), `SibSp`\n",
    "  (OR 0.69) and `LogFare` (OR 1.54).** Their odds-ratio confidence intervals exclude 1, so the direction of each effect is reliable.\n",
    "- **Not individually significant:** `Parch`, `Embarked_Q`, `Embarked_S`, `AgeMissing`, `Pclass_2` — and, notably, `Pclass_3`\n",
    "  (p = 0.11, 95% CI for its OR 0.16–1.20). This is the multicollinearity flagged in Section 1: class, cabin and fare overlap\n",
    "  (`Pclass`–`HasCabin` r = −0.73), so the model cannot cleanly attribute the effect to any one of them and each estimate\n",
    "  gets a wide interval. The joint test below asks the better question — does class matter at all, given everything else?\n",
    "- **Age reversal (confounding):** age was *not* significant alone (p = 0.16) but is strongly significant here (p < 0.001).\n",
    "  Older passengers were more often in 1st class, which helped them; that masked the negative effect of age. Controlling\n",
    "  for class reveals it — a key reason to use a multivariate model instead of one-variable comparisons.\n",
    "- The likelihood-ratio test (p ≈ 2.5 × 10⁻⁶⁷) confirms the model as a whole is far better than an intercept-only model;\n",
    "  McFadden pseudo-R² = 0.364 (values of 0.2–0.4 are conventionally regarded as a good fit for logistic models)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c7917ea8",
   "metadata": {},
   "source": [
    "**Joint Wald test for passenger class.** H₀: both class coefficients (`Pclass_2`, `Pclass_3`) are zero."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 29,
   "id": "40eb366b",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:06.633962Z",
     "iopub.status.busy": "2026-10-25T17:03:06.633776Z",
     "iopub.status.idle": "2026-10-25T17:03:06.662680Z",
     "shell.execute_reply": "2026-10-25T17:03:06.660470Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Wald chi-square = 11.22, df = 2, p-value = 0.0037\n"
     ]
    }
   ],
   "source": [
    "wald = logit.wald_test(\"Pclass_2 = 0, Pclass_3 = 0\", scalar=True)\n",
    "print(f\"Wald chi-square = {float(wald.statistic):.2f}, df = 2, p-value = {float(wald.pvalue):.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "acf1145d",
   "metadata": {},
   "source": [
    "**Interpretation:** p ≈ 0.004 < 0.05, so passenger class **is** jointly significant even after controlling for cabin and\n",
    "fare — class matters; its effect is simply shared with the correlated cabin and fare variables. The practical reading:\n",
    "`Sex_male`, `Age`, `SibSp` and the class/cabin/fare group drive survival, while port of embarkation and number of\n",
    "parents/children aboard add no reliable information."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "31950380",
   "metadata": {},
   "source": [
    "### 5c. Worked example — what the model predicts for typical passengers\n",
    "**Purpose:** translate coefficients into intuitive probabilities."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 30,
   "id": "2f791dc3",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:06.665044Z",
     "iopub.status.busy": "2026-10-25T17:03:06.664831Z",
     "iopub.status.idle": "2026-10-25T17:03:06.697052Z",
     "shell.execute_reply": "2026-10-25T17:03:06.696141Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Age</th>\n",
       "      <th>SibSp</th>\n",
       "      <th>Parch</th>\n",
       "      <th>Fare</th>\n",
       "      <th>Pclass</th>\n",
       "      <th>Sex</th>\n",
       "      <th>Embarked</th>\n",
       "      <th>HasCabin</th>\n",
       "      <th>P(survive)</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>1st-class woman, 30</th>\n",
       "      <td>30</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>80.0</td>\n",
       "      <td>1</td>\n",
       "      <td>female</td>\n",
       "      <td>C</td>\n",
       "      <td>1</td>\n",
       "      <td>0.967</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1st-class man, 30</th>\n",
       "      <td>30</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>80.0</td>\n",
       "      <td>1</td>\n",
       "      <td>male</td>\n",
       "      <td>C</td>\n",
       "      <td>1</td>\n",
       "      <td>0.708</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3rd-class woman, 25</th>\n",
       "      <td>25</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>8.0</td>\n",
       "      <td>3</td>\n",
       "      <td>female</td>\n",
       "      <td>S</td>\n",
       "      <td>0</td>\n",
       "      <td>0.578</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3rd-class man, 25</th>\n",
       "      <td>25</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>8.0</td>\n",
       "      <td>3</td>\n",
       "      <td>male</td>\n",
       "      <td>S</td>\n",
       "      <td>0</td>\n",
       "      <td>0.102</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2nd-class boy, 6</th>\n",
       "      <td>6</td>\n",
       "      <td>1</td>\n",
       "      <td>2</td>\n",
       "      <td>30.0</td>\n",
       "      <td>2</td>\n",
       "      <td>male</td>\n",
       "      <td>S</td>\n",
       "      <td>0</td>\n",
       "      <td>0.322</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                     Age  SibSp  Parch  Fare  Pclass     Sex Embarked  \\\n",
       "1st-class woman, 30   30      0      0  80.0       1  female        C   \n",
       "1st-class man, 30     30      0      0  80.0       1    male        C   \n",
       "3rd-class woman, 25   25      0      0   8.0       3  female        S   \n",
       "3rd-class man, 25     25      0      0   8.0       3    male        S   \n",
       "2nd-class boy, 6       6      1      2  30.0       2    male        S   \n",
       "\n",
       "                     HasCabin  P(survive)  \n",
       "1st-class woman, 30         1       0.967  \n",
       "1st-class man, 30           1       0.708  \n",
       "3rd-class woman, 25         0       0.578  \n",
       "3rd-class man, 25           0       0.102  \n",
       "2nd-class boy, 6            0       0.322  "
      ]
     },
     "execution_count": 30,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "examples = pd.DataFrame([\n",
    "    {\"Age\": 30, \"SibSp\": 0, \"Parch\": 0, \"Fare\": 80.0, \"Pclass\": 1, \"Sex\": \"female\", \"Embarked\": \"C\", \"HasCabin\": 1},\n",
    "    {\"Age\": 30, \"SibSp\": 0, \"Parch\": 0, \"Fare\": 80.0, \"Pclass\": 1, \"Sex\": \"male\",   \"Embarked\": \"C\", \"HasCabin\": 1},\n",
    "    {\"Age\": 25, \"SibSp\": 0, \"Parch\": 0, \"Fare\": 8.0,  \"Pclass\": 3, \"Sex\": \"female\", \"Embarked\": \"S\", \"HasCabin\": 0},\n",
    "    {\"Age\": 25, \"SibSp\": 0, \"Parch\": 0, \"Fare\": 8.0,  \"Pclass\": 3, \"Sex\": \"male\",   \"Embarked\": \"S\", \"HasCabin\": 0},\n",
    "    {\"Age\": 6,  \"SibSp\": 1, \"Parch\": 2, \"Fare\": 30.0, \"Pclass\": 2, \"Sex\": \"male\",   \"Embarked\": \"S\", \"HasCabin\": 0},\n",
    "], index=[\"1st-class woman, 30\", \"1st-class man, 30\", \"3rd-class woman, 25\", \"3rd-class man, 25\", \"2nd-class boy, 6\"])\n",
    "ex = examples.copy()\n",
    "ex[\"LogFare\"] = np.log1p(ex[\"Fare\"]); ex[\"AgeMissing\"] = 0\n",
    "examples[\"P(survive)\"] = model.predict_proba(ex[FEATURES])[:, 1].round(3)\n",
    "examples"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "041c0cae",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Sex dominates at every ticket level:** with an identical 1st-class ticket, the woman gets 0.97 and the man 0.71; with an\n",
    "  identical 3rd-class ticket, the woman gets 0.58 and the man only 0.10.\n",
    "- **Class compounds it:** moving the woman from the 1st-class to the 3rd-class profile drops her probability from 0.97 to 0.58.\n",
    "- **A limitation shows up too:** the 1st-class man's 0.71 is far above the 36.9% actually observed for 1st-class men. This profile\n",
    "  stacks every favourable factor (cabin, high fare, Cherbourg), and because logistic regression *adds* effects in log-odds it\n",
    "  cannot learn that the male penalty was harsher in practice. Similarly, the 6-year-old boy gets only 0.32: age enters\n",
    "  linearly, so the model under-captures the \"children first\" jump for the very young (a `Child` flag or age bands would help)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "007c9061",
   "metadata": {},
   "source": [
    "## 6. Save the Trained Model for Deployment\n",
    "**Purpose:** persist the complete fitted pipeline (preprocessing + logistic regression) with `joblib`. The Streamlit app\n",
    "(`app.py`) loads this single file, so user inputs go through exactly the same imputation, encoding and scaling as during\n",
    "training.\n",
    "\n",
    "Run the app locally with: `streamlit run app.py`"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 31,
   "id": "034ff8de",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:06.698917Z",
     "iopub.status.busy": "2026-10-25T17:03:06.698739Z",
     "iopub.status.idle": "2026-10-25T17:03:06.728046Z",
     "shell.execute_reply": "2026-10-25T17:03:06.726515Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Saved model/titanic_logreg_pipeline.joblib — reload check passed.\n"
     ]
    }
   ],
   "source": [
    "joblib.dump(model, \"model/titanic_logreg_pipeline.joblib\")\n",
    "reloaded = joblib.load(\"model/titanic_logreg_pipeline.joblib\")\n",
    "assert np.allclose(reloaded.predict_proba(X_test), model.predict_proba(X_test))\n",
    "print(\"Saved model/titanic_logreg_pipeline.joblib — reload check passed.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "34d4e710",
   "metadata": {},
   "source": [
    "## 7. Conclusion\n",
    "- **Model performance:** logistic regression predicts Titanic survival with **79.3% accuracy, 0.767 precision, 0.667 recall,\n",
    "  0.713 F1 and 0.833 ROC-AUC** on 179 unseen passengers (cross-validated AUC 0.864 ± 0.020), well above the 61.5%\n",
    "  majority-class baseline, with no sign of over-fitting.\n",
    "- **What determined survival:** sex was decisive (men had ~12× lower odds), followed by the class/cabin/fare group (jointly\n",
    "  significant; 3rd class roughly halved the odds, a cabin record roughly tripled them), age (−39% odds per ~13 years) and number\n",
    "  of siblings/spouses aboard. Port of embarkation and parents/children aboard were not statistically significant once these were\n",
    "  controlled for.\n",
    "- **Key analytical insight:** age looked irrelevant in isolation but was a significant negative factor after controlling for\n",
    "  class — single-variable analysis can hide real effects when features are confounded.\n",
    "- **Limitations:** a small dataset (891 rows); 20% of ages imputed with a single median; logistic regression assumes effects\n",
    "  add linearly in log-odds, so interactions such as the much weaker benefit of being female in 3rd class are only partly\n",
    "  captured; and survival depended on situational factors (location at the time of impact, lifeboat access) not in the data.\n",
    "- **Deployment:** the full pipeline is saved and served through a Streamlit app (`app.py`) that returns a survival probability\n",
    "  for any passenger profile."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.17"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "a8d56dc5",
   "metadata": {},
   "source": [
    "# Case Study 6 — Comparative Analysis of Classification Algorithms for Bank Marketing Prediction\n",
    "\n",
    "| | |\n",
    "|---|---|\n",
    "| **Subject** | Fundamentals of Data Science |\n",
    "| **Course code** | 22CSC59 |\n",
    "| **Faculty** | S. Durga Devi |\n",
    "\n",
    "**Scenario.** A Portuguese bank runs telephone campaigns to sell term deposits. Calling everyone is expensive and most\n",
    "customers say no. The bank wants to know, *before* calling, which customers are likely to subscribe.\n",
    "\n",
    "**Problem statement.** Build and evaluate four classifiers — **Logistic Regression, Decision Tree, K-Nearest Neighbours and\n",
    "Random Forest** — to predict whether a customer will subscribe (`y = yes`). Compare them with accuracy, precision, recall,\n",
    "F1, confusion matrices and ROC-AUC, and recommend the model best suited to the bank's campaign.\n",
    "\n",
    "**Dataset.** UCI *Bank Marketing* — `bank-additional-full.csv` (41,188 calls, May 2008 – Nov 2010, 20 inputs + target),\n",
    "semicolon-delimited. This is the most complete of the four files supplied: it includes every call (not a 10% sample) and five\n",
    "macro-economic indicators (employment variation rate, consumer price/confidence indices, Euribor rate, number employed).\n",
    "\n",
    "**Important design rule.** `duration` (length of the last call) is **excluded** from all models: it is only known *after* the\n",
    "call ends, so a model using it cannot be used to decide whom to call. The UCI documentation explicitly warns about this."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0617292a",
   "metadata": {},
   "source": [
    "## 0. Setup\n",
    "Import libraries and fix the random seed so every number in this notebook (and the report) is reproducible."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "f46ba607",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:10.206649Z",
     "iopub.status.busy": "2026-10-25T17:03:10.206392Z",
     "iopub.status.idle": "2026-10-25T17:03:11.740282Z",
     "shell.execute_reply": "2026-10-25T17:03:11.738828Z"
    }
   },
   "outputs": [],
   "source": [
    "import os, warnings, time\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from scipy import stats\n",
    "\n",
    "from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict\n",
    "from sklearn.compose import ColumnTransformer\n",
    "from sklearn.preprocessing import OneHotEncoder, StandardScaler\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.tree import DecisionTreeClassifier\n",
    "from sklearn.neighbors import KNeighborsClassifier\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,\n",
    "                             average_precision_score, roc_curve, precision_recall_curve,\n",
    "                             confusion_matrix, ConfusionMatrixDisplay)\n",
    "\n",
    "warnings.filterwarnings(\"ignore\")\n",
    "RANDOM_STATE = 42\n",
    "sns.set_theme(style=\"whitegrid\", context=\"notebook\")\n",
    "os.makedirs(\"figures\", exist_ok=True)\n",
    "\n",
    "def savefig(name):\n",
    "    plt.tight_layout()\n",
    "    plt.savefig(f\"figures/{name}.png\", dpi=150, bbox_inches=\"tight\")\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2627ade8",
   "metadata": {},
   "source": [
    "## Task 1 — Data Understanding and Preprocessing\n",
    "### 1.1 Load the data\n",
    "**Purpose:** read the file with the correct delimiter (`;`) and inspect the first 10 records."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "4d3a83ed",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:11.743189Z",
     "iopub.status.busy": "2026-10-25T17:03:11.742791Z",
     "iopub.status.idle": "2026-10-25T17:03:12.101149Z",
     "shell.execute_reply": "2026-10-25T17:03:12.100540Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Dimensions (rows, columns): (41188, 21)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>age</th>\n",
       "      <th>job</th>\n",
       "      <th>marital</th>\n",
       "      <th>education</th>\n",
       "      <th>default</th>\n",
       "      <th>housing</th>\n",
       "      <th>loan</th>\n",
       "      <th>contact</th>\n",
       "      <th>month</th>\n",
       "      <th>day_of_week</th>\n",
       "      <th>...</th>\n",
       "      <th>campaign</th>\n",
       "      <th>pdays</th>\n",
       "      <th>previous</th>\n",
       "      <th>poutcome</th>\n",
       "      <th>emp.var.rate</th>\n",
       "      <th>cons.price.idx</th>\n",
       "      <th>cons.conf.idx</th>\n",
       "      <th>euribor3m</th>\n",
       "      <th>nr.employed</th>\n",
       "      <th>y</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>56</td>\n",
       "      <td>housemaid</td>\n",
       "      <td>married</td>\n",
       "      <td>basic.4y</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>57</td>\n",
       "      <td>services</td>\n",
       "      <td>married</td>\n",
       "      <td>high.school</td>\n",
       "      <td>unknown</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>37</td>\n",
       "      <td>services</td>\n",
       "      <td>married</td>\n",
       "      <td>high.school</td>\n",
       "      <td>no</td>\n",
       "      <td>yes</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>40</td>\n",
       "      <td>admin.</td>\n",
       "      <td>married</td>\n",
       "      <td>basic.6y</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>56</td>\n",
       "      <td>services</td>\n",
       "      <td>married</td>\n",
       "      <td>high.school</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>yes</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>45</td>\n",
       "      <td>services</td>\n",
       "      <td>married</td>\n",
       "      <td>basic.9y</td>\n",
       "      <td>unknown</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>59</td>\n",
       "      <td>admin.</td>\n",
       "      <td>married</td>\n",
       "      <td>professional.course</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>41</td>\n",
       "      <td>blue-collar</td>\n",
       "      <td>married</td>\n",
       "      <td>unknown</td>\n",
       "      <td>unknown</td>\n",
       "      <td>no</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>24</td>\n",
       "      <td>technician</td>\n",
       "      <td>single</td>\n",
       "      <td>professional.course</td>\n",
       "      <td>no</td>\n",
       "      <td>yes</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>25</td>\n",
       "      <td>services</td>\n",
       "      <td>single</td>\n",
       "      <td>high.school</td>\n",
       "      <td>no</td>\n",
       "      <td>yes</td>\n",
       "      <td>no</td>\n",
       "      <td>telephone</td>\n",
       "      <td>may</td>\n",
       "      <td>mon</td>\n",
       "      <td>...</td>\n",
       "      <td>1</td>\n",
       "      <td>999</td>\n",
       "      <td>0</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>1.1</td>\n",
       "      <td>93.994</td>\n",
       "      <td>-36.4</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5191.0</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "<p>10 rows × 21 columns</p>\n",
       "</div>"
      ],
      "text/plain": [
       "   age          job  marital            education  default housing loan  \\\n",
       "0   56    housemaid  married             basic.4y       no      no   no   \n",
       "1   57     services  married          high.school  unknown      no   no   \n",
       "2   37     services  married          high.school       no     yes   no   \n",
       "3   40       admin.  married             basic.6y       no      no   no   \n",
       "4   56     services  married          high.school       no      no  yes   \n",
       "5   45     services  married             basic.9y  unknown      no   no   \n",
       "6   59       admin.  married  professional.course       no      no   no   \n",
       "7   41  blue-collar  married              unknown  unknown      no   no   \n",
       "8   24   technician   single  professional.course       no     yes   no   \n",
       "9   25     services   single          high.school       no     yes   no   \n",
       "\n",
       "     contact month day_of_week  ...  campaign  pdays  previous     poutcome  \\\n",
       "0  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "1  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "2  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "3  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "4  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "5  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "6  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "7  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "8  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "9  telephone   may         mon  ...         1    999         0  nonexistent   \n",
       "\n",
       "  emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.employed   y  \n",
       "0          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "1          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "2          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "3          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "4          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "5          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "6          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "7          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "8          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "9          1.1          93.994          -36.4      4.857       5191.0  no  \n",
       "\n",
       "[10 rows x 21 columns]"
      ]
     },
     "execution_count": 2,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df_raw = pd.read_csv(\"data/bank-additional-full.csv\", sep=\";\")\n",
    "print(\"Dimensions (rows, columns):\", df_raw.shape)\n",
    "df_raw.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "66fd4236",
   "metadata": {},
   "source": [
    "**Observation:** each row is one phone contact with a customer: client profile (`age` … `loan`), details of the current\n",
    "campaign contact (`contact`, `month`, `day_of_week`, `duration`, `campaign`), history from earlier campaigns (`pdays`,\n",
    "`previous`, `poutcome`), economic context at the time of the call, and the outcome `y`."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2d82351a",
   "metadata": {},
   "source": [
    "### 1.2 Data types and descriptive statistics\n",
    "**Purpose:** confirm each column's type and the range / spread of the numeric attributes."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "d8b76cda",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.102971Z",
     "iopub.status.busy": "2026-10-25T17:03:12.102747Z",
     "iopub.status.idle": "2026-10-25T17:03:12.124068Z",
     "shell.execute_reply": "2026-10-25T17:03:12.123047Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>dtype</th>\n",
       "      <th>unique values</th>\n",
       "      <th>example</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>age</th>\n",
       "      <td>int64</td>\n",
       "      <td>78</td>\n",
       "      <td>56</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>job</th>\n",
       "      <td>str</td>\n",
       "      <td>12</td>\n",
       "      <td>housemaid</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>marital</th>\n",
       "      <td>str</td>\n",
       "      <td>4</td>\n",
       "      <td>married</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>education</th>\n",
       "      <td>str</td>\n",
       "      <td>8</td>\n",
       "      <td>basic.4y</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>default</th>\n",
       "      <td>str</td>\n",
       "      <td>3</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>housing</th>\n",
       "      <td>str</td>\n",
       "      <td>3</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>loan</th>\n",
       "      <td>str</td>\n",
       "      <td>3</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>contact</th>\n",
       "      <td>str</td>\n",
       "      <td>2</td>\n",
       "      <td>telephone</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>month</th>\n",
       "      <td>str</td>\n",
       "      <td>10</td>\n",
       "      <td>may</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>day_of_week</th>\n",
       "      <td>str</td>\n",
       "      <td>5</td>\n",
       "      <td>mon</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>duration</th>\n",
       "      <td>int64</td>\n",
       "      <td>1544</td>\n",
       "      <td>261</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>campaign</th>\n",
       "      <td>int64</td>\n",
       "      <td>42</td>\n",
       "      <td>1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>pdays</th>\n",
       "      <td>int64</td>\n",
       "      <td>27</td>\n",
       "      <td>999</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>previous</th>\n",
       "      <td>int64</td>\n",
       "      <td>8</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>poutcome</th>\n",
       "      <td>str</td>\n",
       "      <td>3</td>\n",
       "      <td>nonexistent</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>emp.var.rate</th>\n",
       "      <td>float64</td>\n",
       "      <td>10</td>\n",
       "      <td>1.1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>cons.price.idx</th>\n",
       "      <td>float64</td>\n",
       "      <td>26</td>\n",
       "      <td>93.994</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>cons.conf.idx</th>\n",
       "      <td>float64</td>\n",
       "      <td>26</td>\n",
       "      <td>-36.4</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>euribor3m</th>\n",
       "      <td>float64</td>\n",
       "      <td>316</td>\n",
       "      <td>4.857</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>nr.employed</th>\n",
       "      <td>float64</td>\n",
       "      <td>11</td>\n",
       "      <td>5191.0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>y</th>\n",
       "      <td>str</td>\n",
       "      <td>2</td>\n",
       "      <td>no</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                  dtype  unique values      example\n",
       "age               int64             78           56\n",
       "job                 str             12    housemaid\n",
       "marital             str              4      married\n",
       "education           str              8     basic.4y\n",
       "default             str              3           no\n",
       "housing             str              3           no\n",
       "loan                str              3           no\n",
       "contact             str              2    telephone\n",
       "month               str             10          may\n",
       "day_of_week         str              5          mon\n",
       "duration          int64           1544          261\n",
       "campaign          int64             42            1\n",
       "pdays             int64             27          999\n",
       "previous          int64              8            0\n",
       "poutcome            str              3  nonexistent\n",
       "emp.var.rate    float64             10          1.1\n",
       "cons.price.idx  float64             26       93.994\n",
       "cons.conf.idx   float64             26        -36.4\n",
       "euribor3m       float64            316        4.857\n",
       "nr.employed     float64             11       5191.0\n",
       "y                   str              2           no"
      ]
     },
     "execution_count": 3,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "pd.DataFrame({\"dtype\": df_raw.dtypes.astype(str), \"unique values\": df_raw.nunique(),\n",
    "              \"example\": df_raw.iloc[0]})"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "042c3f43",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.125917Z",
     "iopub.status.busy": "2026-10-25T17:03:12.125727Z",
     "iopub.status.idle": "2026-10-25T17:03:12.161460Z",
     "shell.execute_reply": "2026-10-25T17:03:12.160247Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>count</th>\n",
       "      <th>mean</th>\n",
       "      <th>std</th>\n",
       "      <th>min</th>\n",
       "      <th>25%</th>\n",
       "      <th>50%</th>\n",
       "      <th>75%</th>\n",
       "      <th>max</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>age</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>40.02</td>\n",
       "      <td>10.42</td>\n",
       "      <td>17.00</td>\n",
       "      <td>32.00</td>\n",
       "      <td>38.00</td>\n",
       "      <td>47.00</td>\n",
       "      <td>98.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>duration</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>258.29</td>\n",
       "      <td>259.28</td>\n",
       "      <td>0.00</td>\n",
       "      <td>102.00</td>\n",
       "      <td>180.00</td>\n",
       "      <td>319.00</td>\n",
       "      <td>4918.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>campaign</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>2.57</td>\n",
       "      <td>2.77</td>\n",
       "      <td>1.00</td>\n",
       "      <td>1.00</td>\n",
       "      <td>2.00</td>\n",
       "      <td>3.00</td>\n",
       "      <td>56.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>pdays</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>962.48</td>\n",
       "      <td>186.91</td>\n",
       "      <td>0.00</td>\n",
       "      <td>999.00</td>\n",
       "      <td>999.00</td>\n",
       "      <td>999.00</td>\n",
       "      <td>999.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>previous</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>0.17</td>\n",
       "      <td>0.49</td>\n",
       "      <td>0.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>0.00</td>\n",
       "      <td>7.00</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>emp.var.rate</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>0.08</td>\n",
       "      <td>1.57</td>\n",
       "      <td>-3.40</td>\n",
       "      <td>-1.80</td>\n",
       "      <td>1.10</td>\n",
       "      <td>1.40</td>\n",
       "      <td>1.40</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>cons.price.idx</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>93.58</td>\n",
       "      <td>0.58</td>\n",
       "      <td>92.20</td>\n",
       "      <td>93.08</td>\n",
       "      <td>93.75</td>\n",
       "      <td>93.99</td>\n",
       "      <td>94.77</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>cons.conf.idx</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>-40.50</td>\n",
       "      <td>4.63</td>\n",
       "      <td>-50.80</td>\n",
       "      <td>-42.70</td>\n",
       "      <td>-41.80</td>\n",
       "      <td>-36.40</td>\n",
       "      <td>-26.90</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>euribor3m</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>3.62</td>\n",
       "      <td>1.73</td>\n",
       "      <td>0.63</td>\n",
       "      <td>1.34</td>\n",
       "      <td>4.86</td>\n",
       "      <td>4.96</td>\n",
       "      <td>5.04</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>nr.employed</th>\n",
       "      <td>41188.0</td>\n",
       "      <td>5167.04</td>\n",
       "      <td>72.25</td>\n",
       "      <td>4963.60</td>\n",
       "      <td>5099.10</td>\n",
       "      <td>5191.00</td>\n",
       "      <td>5228.10</td>\n",
       "      <td>5228.10</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                  count     mean     std      min      25%      50%      75%  \\\n",
       "age             41188.0    40.02   10.42    17.00    32.00    38.00    47.00   \n",
       "duration        41188.0   258.29  259.28     0.00   102.00   180.00   319.00   \n",
       "campaign        41188.0     2.57    2.77     1.00     1.00     2.00     3.00   \n",
       "pdays           41188.0   962.48  186.91     0.00   999.00   999.00   999.00   \n",
       "previous        41188.0     0.17    0.49     0.00     0.00     0.00     0.00   \n",
       "emp.var.rate    41188.0     0.08    1.57    -3.40    -1.80     1.10     1.40   \n",
       "cons.price.idx  41188.0    93.58    0.58    92.20    93.08    93.75    93.99   \n",
       "cons.conf.idx   41188.0   -40.50    4.63   -50.80   -42.70   -41.80   -36.40   \n",
       "euribor3m       41188.0     3.62    1.73     0.63     1.34     4.86     4.96   \n",
       "nr.employed     41188.0  5167.04   72.25  4963.60  5099.10  5191.00  5228.10   \n",
       "\n",
       "                    max  \n",
       "age               98.00  \n",
       "duration        4918.00  \n",
       "campaign          56.00  \n",
       "pdays            999.00  \n",
       "previous           7.00  \n",
       "emp.var.rate       1.40  \n",
       "cons.price.idx    94.77  \n",
       "cons.conf.idx    -26.90  \n",
       "euribor3m          5.04  \n",
       "nr.employed     5228.10  "
      ]
     },
     "execution_count": 4,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df_raw.describe().T.round(2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "f746278b",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.163683Z",
     "iopub.status.busy": "2026-10-25T17:03:12.163497Z",
     "iopub.status.idle": "2026-10-25T17:03:12.194563Z",
     "shell.execute_reply": "2026-10-25T17:03:12.193138Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>count</th>\n",
       "      <th>unique</th>\n",
       "      <th>top</th>\n",
       "      <th>freq</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>job</th>\n",
       "      <td>41188</td>\n",
       "      <td>12</td>\n",
       "      <td>admin.</td>\n",
       "      <td>10422</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>marital</th>\n",
       "      <td>41188</td>\n",
       "      <td>4</td>\n",
       "      <td>married</td>\n",
       "      <td>24928</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>education</th>\n",
       "      <td>41188</td>\n",
       "      <td>8</td>\n",
       "      <td>university.degree</td>\n",
       "      <td>12168</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>default</th>\n",
       "      <td>41188</td>\n",
       "      <td>3</td>\n",
       "      <td>no</td>\n",
       "      <td>32588</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>housing</th>\n",
       "      <td>41188</td>\n",
       "      <td>3</td>\n",
       "      <td>yes</td>\n",
       "      <td>21576</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>loan</th>\n",
       "      <td>41188</td>\n",
       "      <td>3</td>\n",
       "      <td>no</td>\n",
       "      <td>33950</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>contact</th>\n",
       "      <td>41188</td>\n",
       "      <td>2</td>\n",
       "      <td>cellular</td>\n",
       "      <td>26144</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>month</th>\n",
       "      <td>41188</td>\n",
       "      <td>10</td>\n",
       "      <td>may</td>\n",
       "      <td>13769</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>day_of_week</th>\n",
       "      <td>41188</td>\n",
       "      <td>5</td>\n",
       "      <td>thu</td>\n",
       "      <td>8623</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>poutcome</th>\n",
       "      <td>41188</td>\n",
       "      <td>3</td>\n",
       "      <td>nonexistent</td>\n",
       "      <td>35563</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>y</th>\n",
       "      <td>41188</td>\n",
       "      <td>2</td>\n",
       "      <td>no</td>\n",
       "      <td>36548</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "             count unique                top   freq\n",
       "job          41188     12             admin.  10422\n",
       "marital      41188      4            married  24928\n",
       "education    41188      8  university.degree  12168\n",
       "default      41188      3                 no  32588\n",
       "housing      41188      3                yes  21576\n",
       "loan         41188      3                 no  33950\n",
       "contact      41188      2           cellular  26144\n",
       "month        41188     10                may  13769\n",
       "day_of_week  41188      5                thu   8623\n",
       "poutcome     41188      3        nonexistent  35563\n",
       "y            41188      2                 no  36548"
      ]
     },
     "execution_count": 5,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df_raw.describe(include=[\"object\", \"str\"]).T"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a816415e",
   "metadata": {},
   "source": [
    "### 1.3 Numerical, categorical and target attributes\n",
    "**Purpose:** classify the attributes so each type gets the right preprocessing."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "da0ec1bd",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.196646Z",
     "iopub.status.busy": "2026-10-25T17:03:12.196370Z",
     "iopub.status.idle": "2026-10-25T17:03:12.203146Z",
     "shell.execute_reply": "2026-10-25T17:03:12.201780Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Numerical (10): ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']\n",
      "Categorical (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']\n",
      "Target: y -> ['no', 'yes']\n"
     ]
    }
   ],
   "source": [
    "TARGET = \"y\"\n",
    "numerical = df_raw.select_dtypes(include=\"number\").columns.tolist()\n",
    "categorical = [c for c in df_raw.columns if c not in numerical and c != TARGET]\n",
    "print(f\"Numerical ({len(numerical)}):\", numerical)\n",
    "print(f\"Categorical ({len(categorical)}):\", categorical)\n",
    "print(\"Target:\", TARGET, \"->\", df_raw[TARGET].unique().tolist())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1136f178",
   "metadata": {},
   "source": [
    "**Observation:** 10 numerical attributes (age, call statistics, campaign history, 5 economic indicators) and 10 categorical\n",
    "attributes (job, marital status, education, credit default, housing/personal loan, contact type, month, weekday, previous\n",
    "outcome). The target `y` is binary: `yes` / `no`."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "95a43084",
   "metadata": {},
   "source": [
    "### 1.4 Missing / unknown entries, duplicates and target distribution\n",
    "**Purpose:** this dataset has no blank cells, but missing information is coded as the text `\"unknown\"` in categorical\n",
    "columns and as `999` in `pdays` (\"never contacted before\"). Both must be detected explicitly."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "b24a70f0",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.204963Z",
     "iopub.status.busy": "2026-10-25T17:03:12.204781Z",
     "iopub.status.idle": "2026-10-25T17:03:12.243047Z",
     "shell.execute_reply": "2026-10-25T17:03:12.242327Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Blank (NaN) cells: 0\n",
      "           unknown count  unknown %\n",
      "default             8597      20.87\n",
      "education           1731       4.20\n",
      "housing              990       2.40\n",
      "loan                 990       2.40\n",
      "job                  330       0.80\n",
      "marital               80       0.19\n",
      "\n",
      "pdays == 999 (never previously contacted): 39673 (96.3%)\n",
      "Exact duplicate rows: 12\n"
     ]
    }
   ],
   "source": [
    "print(\"Blank (NaN) cells:\", int(df_raw.isna().sum().sum()))\n",
    "unknown = (df_raw[categorical] == \"unknown\").sum()\n",
    "unknown_tbl = pd.DataFrame({\"unknown count\": unknown, \"unknown %\": (unknown / len(df_raw) * 100).round(2)})\n",
    "unknown_tbl = unknown_tbl[unknown_tbl[\"unknown count\"] > 0].sort_values(\"unknown count\", ascending=False)\n",
    "print(unknown_tbl)\n",
    "print(f\"\\npdays == 999 (never previously contacted): {(df_raw['pdays'] == 999).sum()} \"\n",
    "      f\"({(df_raw['pdays'] == 999).mean():.1%})\")\n",
    "print(\"Exact duplicate rows:\", df_raw.duplicated().sum())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "9348ff9c",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.244800Z",
     "iopub.status.busy": "2026-10-25T17:03:12.244606Z",
     "iopub.status.idle": "2026-10-25T17:03:12.547090Z",
     "shell.execute_reply": "2026-10-25T17:03:12.545830Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 700x360 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, ax = plt.subplots(figsize=(7, 3.6))\n",
    "bars = ax.bar(unknown_tbl.index, unknown_tbl[\"unknown %\"], color=\"#e67e22\")\n",
    "ax.bar_label(bars, labels=[f\"{p:.1f}%\\n({c:,})\" for c, p in zip(unknown_tbl['unknown count'], unknown_tbl['unknown %'])], padding=2)\n",
    "ax.set_title(\"'Unknown' Entries by Categorical Attribute\")\n",
    "ax.set_xlabel(\"Attribute\")\n",
    "ax.set_ylabel(\"Share of records (%)\")\n",
    "ax.set_ylim(0, 28)\n",
    "savefig(\"cs6_01_unknowns\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "849fca4f",
   "metadata": {},
   "source": [
    "**Interpretation and handling**\n",
    "- **`default` is 20.9% unknown** — far too many rows to drop. Customers who decline to state their credit-default status\n",
    "  may behave differently, so `\"unknown\"` is **kept as its own category** in every column; the model can then learn whether\n",
    "  \"not disclosed\" matters. Imputing the mode would erase that signal and invent information.\n",
    "- `education` (4.2%), `housing`/`loan` (2.4%), `job` (0.8%) and `marital` (0.2%) have small unknown shares, handled the same way.\n",
    "- **`pdays = 999` for 96.3% of customers.** Treated as a number, 999 would look like \"contacted 999 days ago\" — a fake extreme\n",
    "  value. It is replaced by a binary flag `prev_contacted` (1 if contacted in an earlier campaign); the information about\n",
    "  past campaigns is also in `previous` and `poutcome`.\n",
    "- **12 exact duplicate rows** are removed so no call is counted twice (and none can appear in both training and test sets)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "8b8f9d1e",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.549110Z",
     "iopub.status.busy": "2026-10-25T17:03:12.548923Z",
     "iopub.status.idle": "2026-10-25T17:03:12.593102Z",
     "shell.execute_reply": "2026-10-25T17:03:12.591905Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "After removing duplicates: (41176, 21)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>customers</th>\n",
       "      <th>percentage</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>y</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>no</th>\n",
       "      <td>36537</td>\n",
       "      <td>88.73</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>yes</th>\n",
       "      <td>4639</td>\n",
       "      <td>11.27</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "     customers  percentage\n",
       "y                         \n",
       "no       36537       88.73\n",
       "yes       4639       11.27"
      ]
     },
     "execution_count": 9,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "df = df_raw.drop_duplicates().reset_index(drop=True)\n",
    "print(\"After removing duplicates:\", df.shape)\n",
    "counts = df[TARGET].value_counts()\n",
    "pct = df[TARGET].value_counts(normalize=True) * 100\n",
    "target_tbl = pd.DataFrame({\"customers\": counts, \"percentage\": pct.round(2)})\n",
    "target_tbl"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "eff79aea",
   "metadata": {},
   "source": [
    "## Task 2 — Exploratory Data Analysis\n",
    "### 2.1 Target distribution\n",
    "**Purpose:** count and visualise how many customers subscribed."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "7088b575",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.595105Z",
     "iopub.status.busy": "2026-10-25T17:03:12.594924Z",
     "iopub.status.idle": "2026-10-25T17:03:12.823035Z",
     "shell.execute_reply": "2026-10-25T17:03:12.821484Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 600x420 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, ax = plt.subplots(figsize=(6, 4.2))\n",
    "bars = ax.bar([\"Did not subscribe (no)\", \"Subscribed (yes)\"], counts[[\"no\", \"yes\"]].values, color=[\"#95a5a6\", \"#2e86de\"])\n",
    "ax.bar_label(bars, labels=[f\"{c:,} ({p:.1f}%)\" for c, p in zip(counts[[\"no\", \"yes\"]], pct[[\"no\", \"yes\"]])], padding=2)\n",
    "ax.set_title(\"Distribution of the Target Variable (Term-Deposit Subscription)\")\n",
    "ax.set_xlabel(\"Campaign outcome\")\n",
    "ax.set_ylabel(\"Number of customers\")\n",
    "ax.set_ylim(0, counts.max() * 1.12)\n",
    "savefig(\"cs6_02_target_distribution\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6c859acc",
   "metadata": {},
   "source": [
    "**Interpretation:** only **4,639 of 41,176 customers (11.27%) subscribed**; 36,537 (88.73%) did not — about 1 success per 8\n",
    "calls. This severe imbalance has a direct consequence: a \"model\" that always predicts *no* is **88.7% accurate** while finding\n",
    "zero subscribers. Accuracy is therefore a misleading yardstick here, and recall, precision, F1 and ROC-AUC must drive the comparison."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "9392a766",
   "metadata": {},
   "source": [
    "### 2.2 Subscription rates across customer and campaign attributes\n",
    "**Purpose:** find which groups subscribe more often than the 11.3% average."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "e1e3be18",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:12.825939Z",
     "iopub.status.busy": "2026-10-25T17:03:12.825668Z",
     "iopub.status.idle": "2026-10-25T17:03:14.180038Z",
     "shell.execute_reply": "2026-10-25T17:03:14.179373Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1500x1000 with 4 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "df[\"subscribed\"] = (df[TARGET] == \"yes\").astype(int)\n",
    "overall = df[\"subscribed\"].mean() * 100\n",
    "month_order = [\"mar\", \"apr\", \"may\", \"jun\", \"jul\", \"aug\", \"sep\", \"oct\", \"nov\", \"dec\"]\n",
    "\n",
    "fig, axes = plt.subplots(2, 2, figsize=(15, 10))\n",
    "specs = [(\"job\", None, \"Job\"), (\"month\", month_order, \"Month of last contact\"),\n",
    "         (\"poutcome\", [\"nonexistent\", \"failure\", \"success\"], \"Outcome of previous campaign\"),\n",
    "         (\"contact\", None, \"Contact channel\")]\n",
    "for ax, (col, order, label) in zip(axes.ravel(), specs):\n",
    "    g = df.groupby(col)[\"subscribed\"].agg([\"mean\", \"size\"])\n",
    "    g = g.loc[order] if order else g.sort_values(\"mean\")\n",
    "    bars = ax.bar(g.index, g[\"mean\"] * 100, color=sns.color_palette(\"Blues_d\", len(g)))\n",
    "    ax.bar_label(bars, labels=[f\"{m*100:.1f}%\" for m in g[\"mean\"]], fontsize=9, padding=2)\n",
    "    ax.axhline(overall, ls=\"--\", c=\"red\", lw=1.2, label=f\"Overall rate {overall:.1f}%\")\n",
    "    ax.set_title(f\"Subscription Rate by {label}\")\n",
    "    ax.set_xlabel(label)\n",
    "    ax.set_ylabel(\"Subscription rate (%)\")\n",
    "    ax.set_ylim(0, max(g[\"mean\"] * 100) * 1.18)\n",
    "    ax.tick_params(axis=\"x\", rotation=35 if col == \"job\" else 0)\n",
    "    ax.legend(loc=\"upper left\")\n",
    "savefig(\"cs6_03_rates_by_attribute\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "7038cdcd",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:14.182148Z",
     "iopub.status.busy": "2026-10-25T17:03:14.181887Z",
     "iopub.status.idle": "2026-10-25T17:03:14.205463Z",
     "shell.execute_reply": "2026-10-25T17:03:14.204577Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "             rate  customers\n",
      "poutcome                    \n",
      "success      65.1       1373\n",
      "failure      14.2       4252\n",
      "nonexistent   8.8      35551 \n",
      "\n",
      "           rate  customers\n",
      "contact                   \n",
      "cellular   14.7      26135\n",
      "telephone   5.2      15041 \n",
      "\n",
      "       rate  customers\n",
      "month                 \n",
      "mar    50.5        546\n",
      "dec    48.9        182\n",
      "sep    44.9        570\n",
      "oct    43.9        717\n",
      "apr    20.5       2631\n",
      "aug    10.6       6176\n",
      "jun    10.5       5318\n",
      "nov    10.1       4100\n",
      "jul     9.0       7169\n",
      "may     6.4      13767 \n",
      "\n",
      "               rate  customers\n",
      "job                           \n",
      "student        31.4        875\n",
      "retired        25.3       1718\n",
      "unemployed     14.2       1014\n",
      "admin.         13.0      10419\n",
      "unknown        11.2        330\n",
      "management     11.2       2924\n",
      "technician     10.8       6739\n",
      "self-employed  10.5       1421\n",
      "housemaid      10.0       1060\n",
      "entrepreneur    8.5       1456\n",
      "services        8.1       3967\n",
      "blue-collar     6.9       9253 \n",
      "\n"
     ]
    }
   ],
   "source": [
    "for col in [\"poutcome\", \"contact\", \"month\", \"job\"]:\n",
    "    t = df.groupby(col)[\"subscribed\"].agg(rate=\"mean\", customers=\"size\")\n",
    "    t[\"rate\"] = (t[\"rate\"] * 100).round(1)\n",
    "    print(t.sort_values(\"rate\", ascending=False).to_string(), \"\\n\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "31858a63",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Previous campaign outcome is the strongest behavioural signal:** customers whose previous campaign ended in *success*\n",
    "  subscribe at **65.1%** — almost 6× the average — versus 14.2% after a previous *failure* and 8.8% for customers never\n",
    "  contacted before.\n",
    "- **Timing matters a lot:** March, September, October and December reach 44–51%, while May — the busiest month (≈13,800\n",
    "  calls, a third of all contacts) — converts at only 6.4%. Months with few calls convert best, suggesting the bank called\n",
    "  more selectively (or under better economic conditions) in those months.\n",
    "- **Job:** students (31.4%) and retirees (25.3%) subscribe far above average; blue-collar workers (6.9%) and services staff\n",
    "  (8.1%) are the least likely.\n",
    "- **Channel:** mobile (`cellular`) contacts convert at 14.7% versus 5.2% for landline (`telephone`)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "196de36d",
   "metadata": {},
   "source": [
    "### 2.3 Numerical variables vs subscription outcome\n",
    "**Purpose:** compare distributions of key numerical variables for subscribers and non-subscribers."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "id": "cb7566c8",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:14.207843Z",
     "iopub.status.busy": "2026-10-25T17:03:14.207639Z",
     "iopub.status.idle": "2026-10-25T17:03:15.128939Z",
     "shell.execute_reply": "2026-10-25T17:03:15.127903Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1800x460 with 4 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>age</th>\n",
       "      <th>campaign</th>\n",
       "      <th>previous</th>\n",
       "      <th>euribor3m</th>\n",
       "      <th>nr.employed</th>\n",
       "      <th>emp.var.rate</th>\n",
       "      <th>cons.conf.idx</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>y</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>no</th>\n",
       "      <td>38.0</td>\n",
       "      <td>2.0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>4.857</td>\n",
       "      <td>5195.8</td>\n",
       "      <td>1.1</td>\n",
       "      <td>-41.8</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>yes</th>\n",
       "      <td>37.0</td>\n",
       "      <td>2.0</td>\n",
       "      <td>0.0</td>\n",
       "      <td>1.266</td>\n",
       "      <td>5099.1</td>\n",
       "      <td>-1.8</td>\n",
       "      <td>-40.4</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "      age  campaign  previous  euribor3m  nr.employed  emp.var.rate  \\\n",
       "y                                                                     \n",
       "no   38.0       2.0       0.0      4.857       5195.8           1.1   \n",
       "yes  37.0       2.0       0.0      1.266       5099.1          -1.8   \n",
       "\n",
       "     cons.conf.idx  \n",
       "y                   \n",
       "no           -41.8  \n",
       "yes          -40.4  "
      ]
     },
     "execution_count": 13,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "num_show = [(\"age\", \"Age (years)\"), (\"campaign\", \"Contacts in this campaign\"),\n",
    "            (\"euribor3m\", \"3-month Euribor rate (%)\"), (\"nr.employed\", \"Number employed (quarterly index)\")]\n",
    "fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))\n",
    "for ax, (col, label) in zip(axes, num_show):\n",
    "    sns.boxplot(data=df, x=TARGET, y=col, order=[\"no\", \"yes\"], ax=ax, palette=[\"#95a5a6\", \"#2e86de\"],\n",
    "                showfliers=(col != \"campaign\"))\n",
    "    ax.set_title(f\"{label} by Outcome\")\n",
    "    ax.set_xlabel(\"Subscribed?\")\n",
    "    ax.set_ylabel(label)\n",
    "axes[1].set_ylim(0, 12)\n",
    "axes[1].set_title(\"Contacts in this Campaign by Outcome\\n(outliers hidden, axis capped at 12)\")\n",
    "savefig(\"cs6_04_numeric_boxplots\")\n",
    "df.groupby(TARGET)[[\"age\", \"campaign\", \"previous\", \"euribor3m\", \"nr.employed\", \"emp.var.rate\", \"cons.conf.idx\"]].median()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "70090d72",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Economic context separates the classes most clearly.** Subscribers' median Euribor rate is **1.27%** versus **4.86%** for\n",
    "  non-subscribers, and the median number employed is markedly lower for subscribers. When interest rates and employment fell\n",
    "  (2009–2010), customers were more receptive — locking in a guaranteed deposit is more attractive in uncertain times.\n",
    "- **Campaign intensity:** both groups have a median of 2 contacts, but non-subscribers have a longer tail of repeated calls —\n",
    "  calling the same person many times rarely converts.\n",
    "- **Age:** medians are almost identical (37 vs 38); age matters mainly at the extremes (students and retirees, seen above)."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a5e44ee2",
   "metadata": {},
   "source": [
    "#### Subscription rate across the range of the economic indicator and call count\n",
    "**Purpose:** show the *shape* of the relationship (box plots only compare centres)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "b79b5098",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:15.131066Z",
     "iopub.status.busy": "2026-10-25T17:03:15.130866Z",
     "iopub.status.idle": "2026-10-25T17:03:15.634539Z",
     "shell.execute_reply": "2026-10-25T17:03:15.633731Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x440 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))\n",
    "e = df.assign(eur_bin=pd.cut(df[\"euribor3m\"], bins=[0, 1, 1.5, 4, 5.1]))\n",
    "r = e.groupby(\"eur_bin\", observed=True)[\"subscribed\"].agg([\"mean\", \"size\"])\n",
    "axes[0].bar(r.index.astype(str), r[\"mean\"] * 100, color=\"#2e86de\")\n",
    "for i, (m, n) in enumerate(zip(r[\"mean\"], r[\"size\"])):\n",
    "    axes[0].text(i, m * 100 + 1, f\"{m*100:.0f}%\\nn={n:,}\", ha=\"center\", fontsize=8.5)\n",
    "axes[0].set_title(\"Subscription Rate by 3-month Euribor Rate\")\n",
    "axes[0].set_xlabel(\"Euribor 3-month rate band (%)\")\n",
    "axes[0].set_ylabel(\"Subscription rate (%)\")\n",
    "axes[0].set_ylim(0, 70)\n",
    "axes[0].set_xticklabels([\"0–1%\", \"1–1.5%\", \"1.5–4%\", \"4–5.1%\"])\n",
    "\n",
    "c = df.assign(c_bin=df[\"campaign\"].clip(upper=10))\n",
    "r2 = c.groupby(\"c_bin\")[\"subscribed\"].agg([\"mean\", \"size\"])\n",
    "axes[1].plot(r2.index, r2[\"mean\"] * 100, marker=\"o\", color=\"#c0392b\")\n",
    "axes[1].set_xticks(range(1, 11))\n",
    "axes[1].set_xticklabels([str(i) for i in range(1, 10)] + [\"10+\"])\n",
    "axes[1].set_title(\"Subscription Rate by Number of Contacts in this Campaign\")\n",
    "axes[1].set_xlabel(\"Number of contacts during this campaign\")\n",
    "axes[1].set_ylabel(\"Subscription rate (%)\")\n",
    "savefig(\"cs6_05_rate_vs_numeric\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8a591bda",
   "metadata": {},
   "source": [
    "**Interpretation:** when the 3-month Euribor rate was below 1%, **45.7%** of contacted customers subscribed; at 1–1.5% the rate\n",
    "was 14.9%, and above 4% — where two-thirds of all calls were made — only about **5%** subscribed. (The small 1.5–4% band, 296\n",
    "calls made during the rapid 2009 rate cuts, is an exception at 42.9%.) The second chart shows diminishing returns from\n",
    "repeated calls: conversion is highest on the first contact and declines with every additional call. Both are actionable:\n",
    "campaign effort should respond to the rate environment, and call attempts per customer should be capped."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "659b4c2c",
   "metadata": {},
   "source": [
    "### 2.4 Correlation among numerical variables\n",
    "**Purpose:** check linear relationships with the target and redundancy among predictors."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "8410d467",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:15.638154Z",
     "iopub.status.busy": "2026-10-25T17:03:15.637861Z",
     "iopub.status.idle": "2026-10-25T17:03:16.399329Z",
     "shell.execute_reply": "2026-10-25T17:03:16.397377Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 900x750 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/plain": [
       "nr.employed      -0.354669\n",
       "pdays            -0.324948\n",
       "euribor3m        -0.307740\n",
       "emp.var.rate     -0.298289\n",
       "cons.price.idx   -0.136134\n",
       "campaign         -0.066361\n",
       "age               0.030381\n",
       "cons.conf.idx     0.054802\n",
       "previous          0.230202\n",
       "duration          0.405297\n",
       "Name: subscribed, dtype: float64"
      ]
     },
     "execution_count": 15,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "corr = df[numerical + [\"subscribed\"]].corr()\n",
    "fig, ax = plt.subplots(figsize=(9, 7.5))\n",
    "sns.heatmap(corr, annot=True, fmt=\".2f\", cmap=\"coolwarm\", center=0, square=True,\n",
    "            cbar_kws={\"label\": \"Pearson correlation\"}, ax=ax, annot_kws={\"size\": 8})\n",
    "ax.set_title(\"Correlation Matrix of Numerical Attributes and Subscription\")\n",
    "savefig(\"cs6_06_correlation\")\n",
    "corr[\"subscribed\"].drop(\"subscribed\").sort_values()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d6481400",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- `duration` has the highest correlation with subscription (+0.41) — the very reason it must be excluded: it is a\n",
    "  consequence of the call outcome, not a cause known beforehand.\n",
    "- Among usable predictors, `nr.employed` (−0.35), `pdays` (−0.32, i.e. *recently* contacted customers subscribe more),\n",
    "  `euribor3m` (−0.31), `emp.var.rate` (−0.30) and `previous` (+0.23) are the strongest.\n",
    "- **The economic indicators are highly inter-correlated** (`euribor3m`–`emp.var.rate` 0.97, `euribor3m`–`nr.employed` 0.95):\n",
    "  they describe the same macro-economic cycle. This barely affects tree-based models, but it makes individual logistic\n",
    "  regression coefficients unstable, and it is one reason the comparison focuses on predictive metrics rather than coefficients."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "242343dc",
   "metadata": {},
   "source": [
    "### 2.5 Statistical significance of the observed differences\n",
    "**Purpose:** test whether the patterns above are larger than chance.\n",
    "- Chi-square test of independence for categorical attributes (with Cramér's V as effect size: ~0.1 small, ~0.3 medium, ~0.5 large).\n",
    "- Mann–Whitney U test for numerical attributes (non-normal distributions), with the rank-biserial correlation as effect size."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "bb46e774",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:16.401805Z",
     "iopub.status.busy": "2026-10-25T17:03:16.401607Z",
     "iopub.status.idle": "2026-10-25T17:03:16.565672Z",
     "shell.execute_reply": "2026-10-25T17:03:16.564303Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Attribute</th>\n",
       "      <th>Test</th>\n",
       "      <th>Statistic</th>\n",
       "      <th>dof</th>\n",
       "      <th>p-value</th>\n",
       "      <th>Effect size</th>\n",
       "      <th>Significant (α=0.05)</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>nr.employed</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>42551676.5</td>\n",
       "      <td>-</td>\n",
       "      <td>0.00e+00</td>\n",
       "      <td>0.498</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>euribor3m</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>43488310.0</td>\n",
       "      <td>-</td>\n",
       "      <td>0.00e+00</td>\n",
       "      <td>0.487</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>emp.var.rate</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>48017796.5</td>\n",
       "      <td>-</td>\n",
       "      <td>0.00e+00</td>\n",
       "      <td>0.433</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>poutcome</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>4230.1</td>\n",
       "      <td>2</td>\n",
       "      <td>0.00e+00</td>\n",
       "      <td>0.321</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>month</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>3103.0</td>\n",
       "      <td>9</td>\n",
       "      <td>0.00e+00</td>\n",
       "      <td>0.275</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>5</th>\n",
       "      <td>cons.price.idx</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>66018250.5</td>\n",
       "      <td>-</td>\n",
       "      <td>1.44e-135</td>\n",
       "      <td>0.221</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>6</th>\n",
       "      <td>previous</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>103274848.5</td>\n",
       "      <td>-</td>\n",
       "      <td>0.00e+00</td>\n",
       "      <td>0.219</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>7</th>\n",
       "      <td>pdays</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>68346066.0</td>\n",
       "      <td>-</td>\n",
       "      <td>0.00e+00</td>\n",
       "      <td>0.194</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>8</th>\n",
       "      <td>job</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>961.7</td>\n",
       "      <td>11</td>\n",
       "      <td>3.27e-199</td>\n",
       "      <td>0.153</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>9</th>\n",
       "      <td>contact</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>862.1</td>\n",
       "      <td>1</td>\n",
       "      <td>1.72e-189</td>\n",
       "      <td>0.145</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>10</th>\n",
       "      <td>campaign</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>75391969.5</td>\n",
       "      <td>-</td>\n",
       "      <td>3.63e-38</td>\n",
       "      <td>0.110</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>11</th>\n",
       "      <td>default</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>406.6</td>\n",
       "      <td>2</td>\n",
       "      <td>5.22e-89</td>\n",
       "      <td>0.099</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>12</th>\n",
       "      <td>cons.conf.idx</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>91054962.5</td>\n",
       "      <td>-</td>\n",
       "      <td>7.10e-17</td>\n",
       "      <td>0.074</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>13</th>\n",
       "      <td>education</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>192.8</td>\n",
       "      <td>7</td>\n",
       "      <td>3.75e-38</td>\n",
       "      <td>0.068</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>14</th>\n",
       "      <td>marital</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>123.0</td>\n",
       "      <td>3</td>\n",
       "      <td>1.78e-26</td>\n",
       "      <td>0.055</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>15</th>\n",
       "      <td>day_of_week</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>26.1</td>\n",
       "      <td>4</td>\n",
       "      <td>3.09e-05</td>\n",
       "      <td>0.025</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>16</th>\n",
       "      <td>age</td>\n",
       "      <td>Mann-Whitney U</td>\n",
       "      <td>82905027.5</td>\n",
       "      <td>-</td>\n",
       "      <td>1.56e-02</td>\n",
       "      <td>0.022</td>\n",
       "      <td>Yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>17</th>\n",
       "      <td>housing</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>5.7</td>\n",
       "      <td>2</td>\n",
       "      <td>5.66e-02</td>\n",
       "      <td>0.012</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>18</th>\n",
       "      <td>loan</td>\n",
       "      <td>Chi-square</td>\n",
       "      <td>1.1</td>\n",
       "      <td>2</td>\n",
       "      <td>5.77e-01</td>\n",
       "      <td>0.005</td>\n",
       "      <td>No</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "         Attribute            Test    Statistic dof    p-value  Effect size  \\\n",
       "0      nr.employed  Mann-Whitney U   42551676.5   -   0.00e+00        0.498   \n",
       "1        euribor3m  Mann-Whitney U   43488310.0   -   0.00e+00        0.487   \n",
       "2     emp.var.rate  Mann-Whitney U   48017796.5   -   0.00e+00        0.433   \n",
       "3         poutcome      Chi-square       4230.1   2   0.00e+00        0.321   \n",
       "4            month      Chi-square       3103.0   9   0.00e+00        0.275   \n",
       "5   cons.price.idx  Mann-Whitney U   66018250.5   -  1.44e-135        0.221   \n",
       "6         previous  Mann-Whitney U  103274848.5   -   0.00e+00        0.219   \n",
       "7            pdays  Mann-Whitney U   68346066.0   -   0.00e+00        0.194   \n",
       "8              job      Chi-square        961.7  11  3.27e-199        0.153   \n",
       "9          contact      Chi-square        862.1   1  1.72e-189        0.145   \n",
       "10        campaign  Mann-Whitney U   75391969.5   -   3.63e-38        0.110   \n",
       "11         default      Chi-square        406.6   2   5.22e-89        0.099   \n",
       "12   cons.conf.idx  Mann-Whitney U   91054962.5   -   7.10e-17        0.074   \n",
       "13       education      Chi-square        192.8   7   3.75e-38        0.068   \n",
       "14         marital      Chi-square        123.0   3   1.78e-26        0.055   \n",
       "15     day_of_week      Chi-square         26.1   4   3.09e-05        0.025   \n",
       "16             age  Mann-Whitney U   82905027.5   -   1.56e-02        0.022   \n",
       "17         housing      Chi-square          5.7   2   5.66e-02        0.012   \n",
       "18            loan      Chi-square          1.1   2   5.77e-01        0.005   \n",
       "\n",
       "   Significant (α=0.05)  \n",
       "0                   Yes  \n",
       "1                   Yes  \n",
       "2                   Yes  \n",
       "3                   Yes  \n",
       "4                   Yes  \n",
       "5                   Yes  \n",
       "6                   Yes  \n",
       "7                   Yes  \n",
       "8                   Yes  \n",
       "9                   Yes  \n",
       "10                  Yes  \n",
       "11                  Yes  \n",
       "12                  Yes  \n",
       "13                  Yes  \n",
       "14                  Yes  \n",
       "15                  Yes  \n",
       "16                  Yes  \n",
       "17                   No  \n",
       "18                   No  "
      ]
     },
     "execution_count": 16,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "rows = []\n",
    "for col in categorical:\n",
    "    t = pd.crosstab(df[col], df[TARGET])\n",
    "    chi2, p, dof, _ = stats.chi2_contingency(t)\n",
    "    v = np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1)))\n",
    "    rows.append([col, \"Chi-square\", chi2, dof, p, v])\n",
    "for col in [c for c in numerical if c != \"duration\"]:\n",
    "    a, b = df.loc[df.subscribed == 1, col], df.loc[df.subscribed == 0, col]\n",
    "    u, p = stats.mannwhitneyu(a, b)\n",
    "    rbc = 1 - 2 * u / (len(a) * len(b))   # rank-biserial correlation\n",
    "    rows.append([col, \"Mann-Whitney U\", u, \"-\", p, abs(rbc)])\n",
    "tests = pd.DataFrame(rows, columns=[\"Attribute\", \"Test\", \"Statistic\", \"dof\", \"p-value\", \"Effect size\"])\n",
    "tests[\"Significant (α=0.05)\"] = np.where(tests[\"p-value\"] < 0.05, \"Yes\", \"No\")\n",
    "tests[\"p-value\"] = tests[\"p-value\"].map(lambda p: f\"{p:.2e}\")\n",
    "tests[\"Statistic\"] = tests[\"Statistic\"].round(1)\n",
    "tests[\"Effect size\"] = tests[\"Effect size\"].round(3)\n",
    "tests.sort_values(\"Effect size\", ascending=False).reset_index(drop=True)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6c64a497",
   "metadata": {},
   "source": [
    "**Interpretation:** with 41,176 records almost every attribute is \"significant\" (p < 0.05) — large samples detect even tiny\n",
    "differences — so the **effect size** is the informative column. The largest effects are the economic indicators\n",
    "(`nr.employed`, `euribor3m`, `emp.var.rate`) and `poutcome`/`month`/`pdays`, matching the visual analysis. `housing` and\n",
    "`loan` are **not significant** (p > 0.05): having a housing or personal loan does not change the subscription rate, so these\n",
    "attributes are unlikely to help the models."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d545e83f",
   "metadata": {},
   "source": [
    "### 2.6 Three key observations from the EDA\n",
    "1. **Subscription is rare and concentrated.** Only 11.3% subscribe, but rates reach 65% for customers whose previous campaign\n",
    "   succeeded, ~31% for students and ~25% for retirees — a small, identifiable segment holds a disproportionate share of\n",
    "   likely subscribers.\n",
    "2. **The economic environment dominates.** When the 3-month Euribor rate was below 1%, 45.7% of contacted customers\n",
    "   subscribed; above 4% (two-thirds of all calls), about 1 in 20 did. Macro indicators have the largest effect sizes of all attributes.\n",
    "3. **More calls ≠ more sales.** Conversion is highest on the first contact and falls with each repeat; mobile contacts convert\n",
    "   almost 3× better than landline ones."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1a0d6fef",
   "metadata": {},
   "source": [
    "### 1.5 Feature preparation, encoding, scaling and the 80:20 split (Task 1 continued)\n",
    "**Purpose:** build the model inputs.\n",
    "- Drop `duration` (not known before the call) and the helper column `subscribed`.\n",
    "- Replace `pdays` by `prev_contacted` (see 1.4).\n",
    "- Target: `yes` → 1, `no` → 0.\n",
    "- **One-hot encode** all 10 categorical attributes (`unknown` stays a category; `handle_unknown=\"ignore\"` protects against\n",
    "  unseen levels at prediction time).\n",
    "- **Standardise** numerical attributes (mean 0, SD 1). This is essential for KNN (distance-based — otherwise `nr.employed`,\n",
    "  in the thousands, would swamp `age`) and for logistic regression's solver; trees are unaffected but use the same\n",
    "  pipeline so all four models see identical inputs.\n",
    "- **80:20 stratified split** (`random_state = 42`) — the *same* split is used for all four models.\n",
    "\n",
    "All transformations are fitted inside a pipeline on the training data only, so nothing about the test set leaks into training."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "id": "6f1d1100",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:16.568499Z",
     "iopub.status.busy": "2026-10-25T17:03:16.568214Z",
     "iopub.status.idle": "2026-10-25T17:03:16.851827Z",
     "shell.execute_reply": "2026-10-25T17:03:16.850362Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Numerical inputs : ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'prev_contacted']\n",
      "Categorical inputs: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']\n",
      "\n",
      "Training set: (32940, 19)  subscription rate 11.27%\n",
      "Test set    : (8236, 19)  subscription rate 11.27%\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Model inputs after one-hot encoding: 62\n"
     ]
    }
   ],
   "source": [
    "data = df.copy()\n",
    "data[\"prev_contacted\"] = (data[\"pdays\"] != 999).astype(int)\n",
    "y = (data[TARGET] == \"yes\").astype(int)\n",
    "X = data.drop(columns=[TARGET, \"subscribed\", \"duration\", \"pdays\"])\n",
    "\n",
    "num_features = X.select_dtypes(include=\"number\").columns.tolist()\n",
    "cat_features = [c for c in X.columns if c not in num_features]\n",
    "print(\"Numerical inputs :\", num_features)\n",
    "print(\"Categorical inputs:\", cat_features)\n",
    "\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)\n",
    "print(f\"\\nTraining set: {X_train.shape}  subscription rate {y_train.mean():.2%}\")\n",
    "print(f\"Test set    : {X_test.shape}  subscription rate {y_test.mean():.2%}\")\n",
    "\n",
    "preprocessor = ColumnTransformer([\n",
    "    (\"num\", StandardScaler(), num_features),\n",
    "    (\"cat\", OneHotEncoder(handle_unknown=\"ignore\"), cat_features),\n",
    "])\n",
    "n_inputs = preprocessor.fit(X_train).transform(X_train.iloc[:5]).shape[1]\n",
    "print(f\"Model inputs after one-hot encoding: {n_inputs}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "31ded09f",
   "metadata": {},
   "source": [
    "## Task 3 — Classification Model Development\n",
    "**Purpose:** train the four required algorithms under identical conditions.\n",
    "\n",
    "| Model | How it decides | Key hyper-parameter(s) tuned |\n",
    "|---|---|---|\n",
    "| Logistic Regression | weighted sum of inputs → sigmoid → probability | `C` (inverse regularisation strength) |\n",
    "| Decision Tree | sequence of yes/no splits on single features | `max_depth`, `min_samples_leaf` (control over-fitting) |\n",
    "| K-Nearest Neighbours | majority vote of the *k* most similar training customers | `n_neighbors` (k) |\n",
    "| Random Forest | average vote of 300 decision trees, each grown on a bootstrap sample with random feature subsets | `max_depth`, `min_samples_leaf` |\n",
    "\n",
    "**Fair tuning.** Untuned defaults would penalise some models unfairly (e.g. an unrestricted decision tree memorises the training\n",
    "data). Each model therefore gets a small grid search with the **same 5-fold stratified cross-validation on the training set\n",
    "only**, selecting by **ROC-AUC** — a threshold-independent metric that is not fooled by the class imbalance. The test set is\n",
    "used exactly once, for the final comparison."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "fc250e6e",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:03:16.853986Z",
     "iopub.status.busy": "2026-10-25T17:03:16.853792Z",
     "iopub.status.idle": "2026-10-25T17:04:57.641236Z",
     "shell.execute_reply": "2026-10-25T17:04:57.640384Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Model</th>\n",
       "      <th>Best hyper-parameters</th>\n",
       "      <th>CV ROC-AUC (train)</th>\n",
       "      <th>Tuning time (s)</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>Logistic Regression</td>\n",
       "      <td>{'C': 0.1}</td>\n",
       "      <td>0.7900</td>\n",
       "      <td>3.9</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>Decision Tree</td>\n",
       "      <td>{'max_depth': 8, 'min_samples_leaf': 20}</td>\n",
       "      <td>0.7854</td>\n",
       "      <td>7.4</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>KNN</td>\n",
       "      <td>{'n_neighbors': 101}</td>\n",
       "      <td>0.7836</td>\n",
       "      <td>10.0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>Random Forest</td>\n",
       "      <td>{'max_depth': 12, 'min_samples_leaf': 5}</td>\n",
       "      <td>0.7991</td>\n",
       "      <td>79.4</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                 Model                     Best hyper-parameters  \\\n",
       "0  Logistic Regression                                {'C': 0.1}   \n",
       "1        Decision Tree  {'max_depth': 8, 'min_samples_leaf': 20}   \n",
       "2                  KNN                      {'n_neighbors': 101}   \n",
       "3        Random Forest  {'max_depth': 12, 'min_samples_leaf': 5}   \n",
       "\n",
       "   CV ROC-AUC (train)  Tuning time (s)  \n",
       "0              0.7900              3.9  \n",
       "1              0.7854              7.4  \n",
       "2              0.7836             10.0  \n",
       "3              0.7991             79.4  "
      ]
     },
     "execution_count": 18,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n",
    "candidates = {\n",
    "    \"Logistic Regression\": (LogisticRegression(max_iter=2000), {\"clf__C\": [0.01, 0.1, 1]}),\n",
    "    \"Decision Tree\": (DecisionTreeClassifier(random_state=RANDOM_STATE),\n",
    "                      {\"clf__max_depth\": [4, 6, 8, 10], \"clf__min_samples_leaf\": [20, 50, 100]}),\n",
    "    \"KNN\": (KNeighborsClassifier(), {\"clf__n_neighbors\": [25, 51, 101]}),\n",
    "    \"Random Forest\": (RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),\n",
    "                      {\"clf__max_depth\": [8, 12, None], \"clf__min_samples_leaf\": [5, 20]}),\n",
    "}\n",
    "models, tuning = {}, []\n",
    "for name, (clf, grid) in candidates.items():\n",
    "    t0 = time.time()\n",
    "    search = GridSearchCV(Pipeline([(\"prep\", preprocessor), (\"clf\", clf)]), grid, cv=cv,\n",
    "                          scoring=\"roc_auc\", n_jobs=-1)\n",
    "    search.fit(X_train, y_train)\n",
    "    models[name] = search.best_estimator_\n",
    "    tuning.append({\"Model\": name,\n",
    "                   \"Best hyper-parameters\": {k.replace(\"clf__\", \"\"): v for k, v in search.best_params_.items()},\n",
    "                   \"CV ROC-AUC (train)\": round(search.best_score_, 4),\n",
    "                   \"Tuning time (s)\": round(time.time() - t0, 1)})\n",
    "pd.DataFrame(tuning)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f792388d",
   "metadata": {},
   "source": [
    "**Observation:** all four models reach a cross-validated ROC-AUC of roughly 0.78–0.80 on the training folds. The selected\n",
    "settings are moderate: the decision tree is limited in depth with a minimum leaf size (preventing memorisation), KNN prefers\n",
    "a large neighbourhood (k = 101) because individual neighbours are noisy, and the random forest uses depth-limited trees."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6fc121d2",
   "metadata": {},
   "source": [
    "### 3.2 Predictions, metrics and confusion matrix for each model\n",
    "**Purpose:** predict the test set with every model, using the standard 0.5 probability threshold, and record all metrics in\n",
    "one comparison table."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "d4e5bc76",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:04:57.644349Z",
     "iopub.status.busy": "2026-10-25T17:04:57.644133Z",
     "iopub.status.idle": "2026-10-25T17:04:58.921054Z",
     "shell.execute_reply": "2026-10-25T17:04:58.919966Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Baseline accuracy (always predict 'no'): 0.8873\n",
      "Baseline PR-AUC (random ranking) = subscription rate: 0.1127\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Accuracy</th>\n",
       "      <th>Precision</th>\n",
       "      <th>Recall</th>\n",
       "      <th>F1-score</th>\n",
       "      <th>ROC-AUC</th>\n",
       "      <th>PR-AUC</th>\n",
       "      <th>TP</th>\n",
       "      <th>FP</th>\n",
       "      <th>FN</th>\n",
       "      <th>TN</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Model</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Logistic Regression</th>\n",
       "      <td>0.8990</td>\n",
       "      <td>0.6633</td>\n",
       "      <td>0.2101</td>\n",
       "      <td>0.3191</td>\n",
       "      <td>0.8004</td>\n",
       "      <td>0.4533</td>\n",
       "      <td>195</td>\n",
       "      <td>99</td>\n",
       "      <td>733</td>\n",
       "      <td>7209</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Decision Tree</th>\n",
       "      <td>0.8970</td>\n",
       "      <td>0.5901</td>\n",
       "      <td>0.2823</td>\n",
       "      <td>0.3819</td>\n",
       "      <td>0.8051</td>\n",
       "      <td>0.4577</td>\n",
       "      <td>262</td>\n",
       "      <td>182</td>\n",
       "      <td>666</td>\n",
       "      <td>7126</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>KNN</th>\n",
       "      <td>0.8991</td>\n",
       "      <td>0.6678</td>\n",
       "      <td>0.2080</td>\n",
       "      <td>0.3172</td>\n",
       "      <td>0.7953</td>\n",
       "      <td>0.4497</td>\n",
       "      <td>193</td>\n",
       "      <td>96</td>\n",
       "      <td>735</td>\n",
       "      <td>7212</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Random Forest</th>\n",
       "      <td>0.9001</td>\n",
       "      <td>0.6606</td>\n",
       "      <td>0.2328</td>\n",
       "      <td>0.3442</td>\n",
       "      <td>0.8114</td>\n",
       "      <td>0.4828</td>\n",
       "      <td>216</td>\n",
       "      <td>111</td>\n",
       "      <td>712</td>\n",
       "      <td>7197</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                     Accuracy  Precision  Recall  F1-score  ROC-AUC  PR-AUC  \\\n",
       "Model                                                                         \n",
       "Logistic Regression    0.8990     0.6633  0.2101    0.3191   0.8004  0.4533   \n",
       "Decision Tree          0.8970     0.5901  0.2823    0.3819   0.8051  0.4577   \n",
       "KNN                    0.8991     0.6678  0.2080    0.3172   0.7953  0.4497   \n",
       "Random Forest          0.9001     0.6606  0.2328    0.3442   0.8114  0.4828   \n",
       "\n",
       "                      TP   FP   FN    TN  \n",
       "Model                                     \n",
       "Logistic Regression  195   99  733  7209  \n",
       "Decision Tree        262  182  666  7126  \n",
       "KNN                  193   96  735  7212  \n",
       "Random Forest        216  111  712  7197  "
      ]
     },
     "execution_count": 19,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "results, probs, preds = [], {}, {}\n",
    "for name, model in models.items():\n",
    "    p = model.predict_proba(X_test)[:, 1]\n",
    "    yhat = (p >= 0.5).astype(int)\n",
    "    probs[name], preds[name] = p, yhat\n",
    "    tn, fp, fn, tp = confusion_matrix(y_test, yhat).ravel()\n",
    "    results.append({\"Model\": name,\n",
    "                    \"Accuracy\": accuracy_score(y_test, yhat),\n",
    "                    \"Precision\": precision_score(y_test, yhat),\n",
    "                    \"Recall\": recall_score(y_test, yhat),\n",
    "                    \"F1-score\": f1_score(y_test, yhat),\n",
    "                    \"ROC-AUC\": roc_auc_score(y_test, p),\n",
    "                    \"PR-AUC\": average_precision_score(y_test, p),\n",
    "                    \"TP\": tp, \"FP\": fp, \"FN\": fn, \"TN\": tn})\n",
    "comparison = pd.DataFrame(results).set_index(\"Model\")\n",
    "baseline_acc = 1 - y_test.mean()\n",
    "print(f\"Baseline accuracy (always predict 'no'): {baseline_acc:.4f}\")\n",
    "print(f\"Baseline PR-AUC (random ranking) = subscription rate: {y_test.mean():.4f}\")\n",
    "comparison.round(4)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "df4cfe90",
   "metadata": {},
   "source": [
    "## Task 4 — Model Performance Comparison and Visualisation\n",
    "### 4.1 Comparison table (threshold 0.5)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "137bc093",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:04:58.923116Z",
     "iopub.status.busy": "2026-10-25T17:04:58.922910Z",
     "iopub.status.idle": "2026-10-25T17:04:58.982720Z",
     "shell.execute_reply": "2026-10-25T17:04:58.981298Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<style type=\"text/css\">\n",
       "#T_8a91a_row1_col2, #T_8a91a_row1_col3, #T_8a91a_row2_col1, #T_8a91a_row3_col0, #T_8a91a_row3_col4 {\n",
       "  background-color: #cfe8ff;\n",
       "}\n",
       "</style>\n",
       "<table id=\"T_8a91a\">\n",
       "  <thead>\n",
       "    <tr>\n",
       "      <th class=\"blank level0\" >&nbsp;</th>\n",
       "      <th id=\"T_8a91a_level0_col0\" class=\"col_heading level0 col0\" >Accuracy</th>\n",
       "      <th id=\"T_8a91a_level0_col1\" class=\"col_heading level0 col1\" >Precision</th>\n",
       "      <th id=\"T_8a91a_level0_col2\" class=\"col_heading level0 col2\" >Recall</th>\n",
       "      <th id=\"T_8a91a_level0_col3\" class=\"col_heading level0 col3\" >F1-score</th>\n",
       "      <th id=\"T_8a91a_level0_col4\" class=\"col_heading level0 col4\" >ROC-AUC</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th class=\"index_name level0\" >Model</th>\n",
       "      <th class=\"blank col0\" >&nbsp;</th>\n",
       "      <th class=\"blank col1\" >&nbsp;</th>\n",
       "      <th class=\"blank col2\" >&nbsp;</th>\n",
       "      <th class=\"blank col3\" >&nbsp;</th>\n",
       "      <th class=\"blank col4\" >&nbsp;</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th id=\"T_8a91a_level0_row0\" class=\"row_heading level0 row0\" >Logistic Regression</th>\n",
       "      <td id=\"T_8a91a_row0_col0\" class=\"data row0 col0\" >0.899000</td>\n",
       "      <td id=\"T_8a91a_row0_col1\" class=\"data row0 col1\" >0.663300</td>\n",
       "      <td id=\"T_8a91a_row0_col2\" class=\"data row0 col2\" >0.210100</td>\n",
       "      <td id=\"T_8a91a_row0_col3\" class=\"data row0 col3\" >0.319100</td>\n",
       "      <td id=\"T_8a91a_row0_col4\" class=\"data row0 col4\" >0.800400</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th id=\"T_8a91a_level0_row1\" class=\"row_heading level0 row1\" >Decision Tree</th>\n",
       "      <td id=\"T_8a91a_row1_col0\" class=\"data row1 col0\" >0.897000</td>\n",
       "      <td id=\"T_8a91a_row1_col1\" class=\"data row1 col1\" >0.590100</td>\n",
       "      <td id=\"T_8a91a_row1_col2\" class=\"data row1 col2\" >0.282300</td>\n",
       "      <td id=\"T_8a91a_row1_col3\" class=\"data row1 col3\" >0.381900</td>\n",
       "      <td id=\"T_8a91a_row1_col4\" class=\"data row1 col4\" >0.805100</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th id=\"T_8a91a_level0_row2\" class=\"row_heading level0 row2\" >KNN</th>\n",
       "      <td id=\"T_8a91a_row2_col0\" class=\"data row2 col0\" >0.899100</td>\n",
       "      <td id=\"T_8a91a_row2_col1\" class=\"data row2 col1\" >0.667800</td>\n",
       "      <td id=\"T_8a91a_row2_col2\" class=\"data row2 col2\" >0.208000</td>\n",
       "      <td id=\"T_8a91a_row2_col3\" class=\"data row2 col3\" >0.317200</td>\n",
       "      <td id=\"T_8a91a_row2_col4\" class=\"data row2 col4\" >0.795300</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th id=\"T_8a91a_level0_row3\" class=\"row_heading level0 row3\" >Random Forest</th>\n",
       "      <td id=\"T_8a91a_row3_col0\" class=\"data row3 col0\" >0.900100</td>\n",
       "      <td id=\"T_8a91a_row3_col1\" class=\"data row3 col1\" >0.660600</td>\n",
       "      <td id=\"T_8a91a_row3_col2\" class=\"data row3 col2\" >0.232800</td>\n",
       "      <td id=\"T_8a91a_row3_col3\" class=\"data row3 col3\" >0.344200</td>\n",
       "      <td id=\"T_8a91a_row3_col4\" class=\"data row3 col4\" >0.811400</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n"
      ],
      "text/plain": [
       "<pandas.io.formats.style.Styler at 0x7fc460254590>"
      ]
     },
     "execution_count": 20,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "comparison[[\"Accuracy\", \"Precision\", \"Recall\", \"F1-score\", \"ROC-AUC\"]].round(4).style.highlight_max(axis=0, color=\"#cfe8ff\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4e007425",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **Accuracy is nearly useless for ranking here:** all four models score ≈ 0.90, only about one point above the 0.887\n",
    "  \"always say no\" baseline.\n",
    "- **Recall is low for every model at the 0.5 threshold (0.21–0.28):** with only 11% positives, few customers ever reach a\n",
    "  50% predicted probability, so the models default to \"no\" and miss roughly three out of four subscribers.\n",
    "- **Precision is ≈ 0.59–0.67:** when a model does flag a customer, roughly 6 in 10 of those calls convert — about 5–6× the\n",
    "  11.3% base rate of calling at random.\n",
    "- **Ranking ability (ROC-AUC):** Random Forest is highest (0.811), followed by Decision Tree (0.805), Logistic Regression\n",
    "  (0.800) and KNN (0.795). Random Forest also has the highest PR-AUC (0.483 vs a 0.113 random baseline).\n",
    "- **F1:** Decision Tree is best at the 0.5 threshold (0.382) because its leaf probabilities push more customers over 0.5\n",
    "  (recall 0.282, the highest); Random Forest is second (0.344). This is a threshold effect, not better ranking — see 4.4–4.6."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0287c011",
   "metadata": {},
   "source": [
    "### 4.2 Grouped bar chart of the four evaluation metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 21,
   "id": "ba582e4f",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:04:58.985108Z",
     "iopub.status.busy": "2026-10-25T17:04:58.984903Z",
     "iopub.status.idle": "2026-10-25T17:04:59.398622Z",
     "shell.execute_reply": "2026-10-25T17:04:59.397734Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1100x500 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "mcols = [\"Accuracy\", \"Precision\", \"Recall\", \"F1-score\"]\n",
    "plot_df = comparison[mcols]\n",
    "x = np.arange(len(mcols)); w = 0.2\n",
    "fig, ax = plt.subplots(figsize=(11, 5))\n",
    "colors = [\"#2e86de\", \"#e67e22\", \"#8e44ad\", \"#27ae60\"]\n",
    "for i, (name, row) in enumerate(plot_df.iterrows()):\n",
    "    b = ax.bar(x + (i - 1.5) * w, row.values, w, label=name, color=colors[i])\n",
    "    ax.bar_label(b, fmt=\"%.2f\", fontsize=8, padding=1)\n",
    "ax.axhline(baseline_acc, ls=\"--\", c=\"grey\", lw=1, label=f\"Accuracy baseline ({baseline_acc:.3f})\")\n",
    "ax.set_xticks(x); ax.set_xticklabels(mcols)\n",
    "ax.set_title(\"Comparison of Evaluation Metrics Across the Four Classifiers (Test Set, Threshold 0.5)\")\n",
    "ax.set_xlabel(\"Evaluation metric\")\n",
    "ax.set_ylabel(\"Score\")\n",
    "ax.set_ylim(0, 1.05)\n",
    "ax.legend(ncol=3, loc=\"upper right\", fontsize=9)\n",
    "savefig(\"cs6_07_metric_comparison\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c8276842",
   "metadata": {},
   "source": [
    "**Interpretation:** the accuracy bars are virtually identical and sit just above the baseline line, while recall and F1 show\n",
    "the real differences — and the real weakness. The chart makes the brief's warning visible: choosing by accuracy would treat the\n",
    "four models as equal and hide that each misses most subscribers at the default threshold."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "725c513a",
   "metadata": {},
   "source": [
    "### 4.3 Confusion matrix for each classifier"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 22,
   "id": "ae3e6ead",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:04:59.400875Z",
     "iopub.status.busy": "2026-10-25T17:04:59.400630Z",
     "iopub.status.idle": "2026-10-25T17:05:00.042698Z",
     "shell.execute_reply": "2026-10-25T17:05:00.041078Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 2000x460 with 4 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>TP</th>\n",
       "      <th>FN</th>\n",
       "      <th>FP</th>\n",
       "      <th>TN</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Model</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Logistic Regression</th>\n",
       "      <td>195</td>\n",
       "      <td>733</td>\n",
       "      <td>99</td>\n",
       "      <td>7209</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Decision Tree</th>\n",
       "      <td>262</td>\n",
       "      <td>666</td>\n",
       "      <td>182</td>\n",
       "      <td>7126</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>KNN</th>\n",
       "      <td>193</td>\n",
       "      <td>735</td>\n",
       "      <td>96</td>\n",
       "      <td>7212</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Random Forest</th>\n",
       "      <td>216</td>\n",
       "      <td>712</td>\n",
       "      <td>111</td>\n",
       "      <td>7197</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                      TP   FN   FP    TN\n",
       "Model                                   \n",
       "Logistic Regression  195  733   99  7209\n",
       "Decision Tree        262  666  182  7126\n",
       "KNN                  193  735   96  7212\n",
       "Random Forest        216  712  111  7197"
      ]
     },
     "execution_count": 22,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))\n",
    "for ax, name in zip(axes, models):\n",
    "    cm = confusion_matrix(y_test, preds[name])\n",
    "    ConfusionMatrixDisplay(cm, display_labels=[\"No\", \"Yes\"]).plot(ax=ax, cmap=\"Blues\", colorbar=False, values_format=\",\")\n",
    "    ax.grid(False)\n",
    "    ax.set_title(f\"{name}\")\n",
    "    ax.set_xlabel(\"Predicted subscription\")\n",
    "    ax.set_ylabel(\"Actual subscription\")\n",
    "fig.suptitle(\"Confusion Matrices on the Test Set (8,236 customers, threshold 0.5)\", y=1.03, fontsize=14)\n",
    "savefig(\"cs6_08_confusion_matrices\")\n",
    "comparison[[\"TP\", \"FN\", \"FP\", \"TN\"]]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "89e89946",
   "metadata": {},
   "source": [
    "**Interpretation (928 actual subscribers in the test set)**\n",
    "- **False negatives dominate the errors.** Every model misses 666–735 of the 928 subscribers (bottom-left cell). For the bank\n",
    "  a false negative is a *lost sale*: a willing customer who is never called.\n",
    "- **False positives are few** (96–182 of 7,308 non-subscribers): customers called unnecessarily. A false positive costs only one wasted call,\n",
    "  so it is much cheaper than a false negative.\n",
    "- Decision Tree finds the most subscribers (262) at the cost of the most false positives (182); Random Forest finds 216 with\n",
    "  111 false positives; Logistic Regression and KNN are the most conservative (195 and 193 found).\n",
    "- Given this cost asymmetry, the default threshold is too strict for the business — addressed in Section 4.6."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "184788a7",
   "metadata": {},
   "source": [
    "### 4.4 ROC curves and ROC-AUC\n",
    "**Purpose:** compare ranking quality across all thresholds."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 23,
   "id": "63c02f13",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:00.045829Z",
     "iopub.status.busy": "2026-10-25T17:05:00.045635Z",
     "iopub.status.idle": "2026-10-25T17:05:00.463845Z",
     "shell.execute_reply": "2026-10-25T17:05:00.462922Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 700x600 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, ax = plt.subplots(figsize=(7, 6))\n",
    "for (name, p), col in zip(probs.items(), colors):\n",
    "    fpr, tpr, _ = roc_curve(y_test, p)\n",
    "    ax.plot(fpr, tpr, lw=2, color=col, label=f\"{name} (AUC = {roc_auc_score(y_test, p):.3f})\")\n",
    "ax.plot([0, 1], [0, 1], \"--\", c=\"grey\", label=\"Random classifier (AUC = 0.500)\")\n",
    "ax.set_title(\"ROC Curves of the Four Classifiers (Test Set)\")\n",
    "ax.set_xlabel(\"False positive rate (non-subscribers wrongly flagged)\")\n",
    "ax.set_ylabel(\"True positive rate / recall (subscribers found)\")\n",
    "ax.legend(loc=\"lower right\")\n",
    "savefig(\"cs6_09_roc_curves\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "721751de",
   "metadata": {},
   "source": [
    "**Interpretation:** the four curves are close together and all well above the diagonal. Random Forest's curve lies on or above\n",
    "the others over most of the range (AUC 0.811). The curves rise steeply at the left: about 60–65% of subscribers can be found\n",
    "while wrongly flagging only ~10% of non-subscribers. After that elbow they flatten — each additional subscriber costs many more\n",
    "false positives, because the remaining subscribers look like typical non-subscribers on pre-call information alone. No model\n",
    "breaks through that ceiling, which indicates the limit is in the available data, not the algorithm."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "ad8f377b",
   "metadata": {},
   "source": [
    "### 4.5 Precision–recall curves (imbalance-aware view)\n",
    "**Purpose:** ROC curves can look optimistic when negatives vastly outnumber positives; the precision–recall curve focuses on the\n",
    "rare positive class."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 24,
   "id": "2509d842",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:00.466361Z",
     "iopub.status.busy": "2026-10-25T17:05:00.466064Z",
     "iopub.status.idle": "2026-10-25T17:05:00.914384Z",
     "shell.execute_reply": "2026-10-25T17:05:00.913193Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 700x550 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, ax = plt.subplots(figsize=(7, 5.5))\n",
    "for (name, p), col in zip(probs.items(), colors):\n",
    "    pr, rc, _ = precision_recall_curve(y_test, p)\n",
    "    ax.plot(rc, pr, lw=2, color=col, label=f\"{name} (AP = {average_precision_score(y_test, p):.3f})\")\n",
    "ax.axhline(y_test.mean(), ls=\"--\", c=\"grey\", label=f\"Random (AP = {y_test.mean():.3f})\")\n",
    "ax.set_title(\"Precision–Recall Curves of the Four Classifiers (Test Set)\")\n",
    "ax.set_xlabel(\"Recall (share of subscribers found)\")\n",
    "ax.set_ylabel(\"Precision (share of flagged customers who subscribe)\")\n",
    "ax.set_ylim(0, 1.02)\n",
    "ax.legend(loc=\"upper right\")\n",
    "savefig(\"cs6_10_pr_curves\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6253154e",
   "metadata": {},
   "source": [
    "**Interpretation:** at 20% recall every model's call list is ≈ 70% precise; at 40% recall ≈ 50–55%; at 60% recall still\n",
    "≈ 45% (four times the base rate); beyond ~65% recall precision drops sharply towards the 0.113 base rate — the same elbow seen\n",
    "in the ROC curves. Random Forest has the highest average precision (0.483), i.e. it\n",
    "keeps call lists \"cleanest\" across most recall levels. This confirms the ROC ranking with an imbalance-robust metric."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f7338036",
   "metadata": {},
   "source": [
    "### 4.5b Is the difference between models statistically reliable?\n",
    "**Purpose:** the ROC-AUC gaps are small (≈ 0.01), so before declaring a winner we check whether they could be chance. A paired\n",
    "bootstrap resamples the 8,236 test customers 1,000 times, recomputes every model's AUC on each resample, and reports a 95%\n",
    "confidence interval for the AUC difference between Random Forest and each other model. If the interval excludes 0, Random\n",
    "Forest is reliably better on this data."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 25,
   "id": "f1585285",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:00.917127Z",
     "iopub.status.busy": "2026-10-25T17:05:00.916879Z",
     "iopub.status.idle": "2026-10-25T17:05:17.012374Z",
     "shell.execute_reply": "2026-10-25T17:05:17.010804Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Mean AUC gain of RF</th>\n",
       "      <th>95% CI low</th>\n",
       "      <th>95% CI high</th>\n",
       "      <th>RF better in % of resamples</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Logistic Regression</th>\n",
       "      <td>0.0109</td>\n",
       "      <td>0.0044</td>\n",
       "      <td>0.0175</td>\n",
       "      <td>100.0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Decision Tree</th>\n",
       "      <td>0.0063</td>\n",
       "      <td>-0.0003</td>\n",
       "      <td>0.0135</td>\n",
       "      <td>96.9</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>KNN</th>\n",
       "      <td>0.0161</td>\n",
       "      <td>0.0082</td>\n",
       "      <td>0.0248</td>\n",
       "      <td>100.0</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                     Mean AUC gain of RF  95% CI low  95% CI high  \\\n",
       "Logistic Regression               0.0109      0.0044       0.0175   \n",
       "Decision Tree                     0.0063     -0.0003       0.0135   \n",
       "KNN                               0.0161      0.0082       0.0248   \n",
       "\n",
       "                     RF better in % of resamples  \n",
       "Logistic Regression                        100.0  \n",
       "Decision Tree                               96.9  \n",
       "KNN                                        100.0  "
      ]
     },
     "execution_count": 25,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "rng = np.random.default_rng(RANDOM_STATE)\n",
    "yt = y_test.values\n",
    "n = len(yt)\n",
    "diffs = {m: [] for m in models if m != \"Random Forest\"}\n",
    "for _ in range(1000):\n",
    "    idx = rng.integers(0, n, n)\n",
    "    if yt[idx].sum() == 0:\n",
    "        continue\n",
    "    rf_auc = roc_auc_score(yt[idx], probs[\"Random Forest\"][idx])\n",
    "    for m in diffs:\n",
    "        diffs[m].append(rf_auc - roc_auc_score(yt[idx], probs[m][idx]))\n",
    "boot = pd.DataFrame({m: {\"Mean AUC gain of RF\": np.mean(d), \"95% CI low\": np.percentile(d, 2.5),\n",
    "                         \"95% CI high\": np.percentile(d, 97.5), \"RF better in % of resamples\": np.mean(np.array(d) > 0) * 100}\n",
    "                     for m, d in diffs.items()}).T\n",
    "boot.round(4)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "75b15073",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- **vs Logistic Regression:** RF's AUC is higher by 0.011 (95% CI 0.004 to 0.018) — reliably better.\n",
    "- **vs KNN:** higher by 0.016 (95% CI 0.008 to 0.025) — reliably better.\n",
    "- **vs Decision Tree:** higher by 0.006 (95% CI −0.0003 to 0.0135; RF better in 96.9% of resamples) — **borderline, not\n",
    "  significant at the 5% level.** The tree is a close second on ranking.\n",
    "- Practical reading: Random Forest is the best ranker, but the margin is modest; tie-breakers (PR-AUC, stability) are used below."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "31feedbe",
   "metadata": {},
   "source": [
    "### 4.6 Selecting the best model and a business-appropriate threshold\n",
    "**Purpose:** the brief says not to choose by accuracy alone. Selection criteria, in order:\n",
    "1. **ROC-AUC and PR-AUC** — ranking quality, independent of the threshold (the threshold can be moved afterwards).\n",
    "2. **F1 / recall at a sensible operating point** — because a missed subscriber (false negative) costs more than an extra call.\n",
    "\n",
    "**Random Forest** ranks first on both ROC-AUC and PR-AUC. Its threshold is then chosen **without touching the test set**: 5-fold\n",
    "out-of-fold predictions on the *training* data are used to find the probability cut-off that maximises F1, and that cut-off is\n",
    "then applied once to the test set."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 26,
   "id": "6edbcfb6",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:17.014677Z",
     "iopub.status.busy": "2026-10-25T17:05:17.014479Z",
     "iopub.status.idle": "2026-10-25T17:05:30.492044Z",
     "shell.execute_reply": "2026-10-25T17:05:30.490953Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Best model by ROC-AUC: Random Forest\n",
      "Threshold maximising F1 on training out-of-fold predictions: 0.21\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Threshold 0.50</th>\n",
       "      <th>Threshold 0.21</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Accuracy</th>\n",
       "      <td>0.9001</td>\n",
       "      <td>0.8781</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Precision</th>\n",
       "      <td>0.6606</td>\n",
       "      <td>0.4681</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Recall</th>\n",
       "      <td>0.2328</td>\n",
       "      <td>0.6013</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>F1-score</th>\n",
       "      <td>0.3442</td>\n",
       "      <td>0.5264</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "           Threshold 0.50  Threshold 0.21\n",
       "Accuracy           0.9001          0.8781\n",
       "Precision          0.6606          0.4681\n",
       "Recall             0.2328          0.6013\n",
       "F1-score           0.3442          0.5264"
      ]
     },
     "execution_count": 26,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "best_name = comparison[\"ROC-AUC\"].idxmax()\n",
    "best = models[best_name]\n",
    "oof = cross_val_predict(best, X_train, y_train, cv=cv, method=\"predict_proba\", n_jobs=-1)[:, 1]\n",
    "ths = np.round(np.arange(0.05, 0.71, 0.01), 2)\n",
    "f1_oof = [f1_score(y_train, (oof >= t).astype(int)) for t in ths]\n",
    "best_t = float(ths[int(np.argmax(f1_oof))])\n",
    "print(f\"Best model by ROC-AUC: {best_name}\")\n",
    "print(f\"Threshold maximising F1 on training out-of-fold predictions: {best_t:.2f}\")\n",
    "\n",
    "p = probs[best_name]\n",
    "yhat_t = (p >= best_t).astype(int)\n",
    "tuned = pd.DataFrame({\n",
    "    \"Threshold 0.50\": [accuracy_score(y_test, preds[best_name]), precision_score(y_test, preds[best_name]),\n",
    "                       recall_score(y_test, preds[best_name]), f1_score(y_test, preds[best_name])],\n",
    "    f\"Threshold {best_t:.2f}\": [accuracy_score(y_test, yhat_t), precision_score(y_test, yhat_t),\n",
    "                                recall_score(y_test, yhat_t), f1_score(y_test, yhat_t)],\n",
    "}, index=[\"Accuracy\", \"Precision\", \"Recall\", \"F1-score\"]).round(4)\n",
    "tuned"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 27,
   "id": "e095c66d",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:30.493976Z",
     "iopub.status.busy": "2026-10-25T17:05:30.493785Z",
     "iopub.status.idle": "2026-10-25T17:05:30.976113Z",
     "shell.execute_reply": "2026-10-25T17:05:30.974853Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x480 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "TP=558  FN=370  FP=634  TN=6674\n"
     ]
    }
   ],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))\n",
    "axes[0].plot(ths, f1_oof, color=\"#27ae60\", lw=2)\n",
    "axes[0].axvline(best_t, ls=\"--\", c=\"red\", label=f\"Chosen threshold = {best_t:.2f}\")\n",
    "axes[0].axvline(0.5, ls=\":\", c=\"grey\", label=\"Default threshold = 0.50\")\n",
    "axes[0].set_title(f\"{best_name}: F1 vs Threshold (Training Out-of-Fold)\")\n",
    "axes[0].set_xlabel(\"Probability threshold for predicting 'subscribe'\")\n",
    "axes[0].set_ylabel(\"F1-score\")\n",
    "axes[0].legend()\n",
    "cm_t = confusion_matrix(y_test, yhat_t)\n",
    "ConfusionMatrixDisplay(cm_t, display_labels=[\"No\", \"Yes\"]).plot(ax=axes[1], cmap=\"Greens\", colorbar=False, values_format=\",\")\n",
    "axes[1].grid(False)\n",
    "axes[1].set_title(f\"{best_name} Confusion Matrix at Threshold {best_t:.2f} (Test Set)\")\n",
    "axes[1].set_xlabel(\"Predicted subscription\")\n",
    "axes[1].set_ylabel(\"Actual subscription\")\n",
    "savefig(\"cs6_11_threshold_tuning\")\n",
    "tn, fp, fn, tp = cm_t.ravel()\n",
    "print(f\"TP={tp}  FN={fn}  FP={fp}  TN={tn}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "41ff5097",
   "metadata": {},
   "source": [
    "**Interpretation**\n",
    "- On the training folds, F1 peaks at a threshold of **0.21** — well below 0.5, as expected when only 11% of customers are positive.\n",
    "- Applied once to the test set, the tuned Random Forest **finds 558 of the 928 subscribers (recall 0.601, up from 0.233)** while\n",
    "  keeping **precision at 0.468** — almost half of all flagged customers subscribe, **4.2× the 11.3% base rate**. F1 rises from\n",
    "  0.344 to **0.526**.\n",
    "- It flags 1,192 of 8,236 customers (14.5%). Missed subscribers fall from 712 to 370; false positives rise from 111 to 634.\n",
    "  For the bank this is the right trade: 523 extra unnecessary calls buy 342 additional term-deposit sales.\n",
    "- **Accuracy falls from 0.900 to 0.878** even though the model is far more useful — a concrete demonstration of why accuracy\n",
    "  must not be the selection criterion on imbalanced data."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d8997348",
   "metadata": {},
   "source": [
    "### 4.7 Cumulative gains — how many subscribers does a targeted call list capture?\n",
    "**Purpose:** translate model quality into a campaign decision: if the bank calls only the top X% of customers ranked by the model,\n",
    "what share of all subscribers does it reach?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 28,
   "id": "d2657fbf",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:30.978044Z",
     "iopub.status.busy": "2026-10-25T17:05:30.977864Z",
     "iopub.status.idle": "2026-10-25T17:05:31.298553Z",
     "shell.execute_reply": "2026-10-25T17:05:31.296704Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 750x550 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Top 10%</th>\n",
       "      <th>Top 20%</th>\n",
       "      <th>Top 30%</th>\n",
       "      <th>Top 50%</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Model</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Logistic Regression</th>\n",
       "      <td>44.8</td>\n",
       "      <td>64.1</td>\n",
       "      <td>72.6</td>\n",
       "      <td>82.1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Decision Tree</th>\n",
       "      <td>45.3</td>\n",
       "      <td>65.7</td>\n",
       "      <td>72.6</td>\n",
       "      <td>84.3</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>KNN</th>\n",
       "      <td>43.5</td>\n",
       "      <td>64.8</td>\n",
       "      <td>71.6</td>\n",
       "      <td>82.5</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Random Forest</th>\n",
       "      <td>46.2</td>\n",
       "      <td>65.3</td>\n",
       "      <td>73.3</td>\n",
       "      <td>83.8</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                     Top 10%  Top 20%  Top 30%  Top 50%\n",
       "Model                                                  \n",
       "Logistic Regression     44.8     64.1     72.6     82.1\n",
       "Decision Tree           45.3     65.7     72.6     84.3\n",
       "KNN                     43.5     64.8     71.6     82.5\n",
       "Random Forest           46.2     65.3     73.3     83.8"
      ]
     },
     "execution_count": 28,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "fig, ax = plt.subplots(figsize=(7.5, 5.5))\n",
    "gains_rows = []\n",
    "for (name, pr_), col in zip(probs.items(), colors):\n",
    "    order = np.argsort(-pr_)\n",
    "    cum = np.cumsum(y_test.values[order]) / y_test.sum()\n",
    "    frac = np.arange(1, len(cum) + 1) / len(cum)\n",
    "    ax.plot(frac * 100, cum * 100, color=col, lw=2, label=name)\n",
    "    gains_rows.append({\"Model\": name, **{f\"Top {k}%\": round(cum[int(len(cum) * k / 100) - 1] * 100, 1) for k in (10, 20, 30, 50)}})\n",
    "ax.plot([0, 100], [0, 100], \"--\", c=\"grey\", label=\"Random calling\")\n",
    "ax.set_title(\"Cumulative Gains Chart: Subscribers Captured vs Customers Called\")\n",
    "ax.set_xlabel(\"Customers called, ranked by model score (%)\")\n",
    "ax.set_ylabel(\"Subscribers captured (%)\")\n",
    "ax.legend(loc=\"lower right\")\n",
    "savefig(\"cs6_12_cumulative_gains\")\n",
    "gains = pd.DataFrame(gains_rows).set_index(\"Model\")\n",
    "gains"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8939515b",
   "metadata": {},
   "source": [
    "**Interpretation:** calling customers in the order Random Forest ranks them, the bank reaches **46.2% of all subscribers by\n",
    "calling only the top 10%** of customers, **65.3% with the top 20%**, and 73.3% with the top 30% — versus 10%, 20% and 30% for\n",
    "random calling (a **4.6× lift** in the top decile). All four models give similar gains curves; Random Forest leads in the\n",
    "top decile, where call budgets are tightest. Beyond ~30% of the list the curves flatten: the last subscribers are expensive to reach."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5ffdc079",
   "metadata": {},
   "source": [
    "### 4.8 What drives the Random Forest's predictions?\n",
    "**Purpose:** check that the chosen model relies on sensible signals (impurity-based feature importance, grouped back to the\n",
    "original attributes by summing their one-hot columns)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 29,
   "id": "13a901ff",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:31.302481Z",
     "iopub.status.busy": "2026-10-25T17:05:31.302232Z",
     "iopub.status.idle": "2026-10-25T17:05:31.875002Z",
     "shell.execute_reply": "2026-10-25T17:05:31.873613Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 800x600 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "data": {
      "text/plain": [
       "euribor3m         16.0\n",
       "nr.employed       14.2\n",
       "poutcome          12.2\n",
       "prev_contacted     8.9\n",
       "emp.var.rate       7.0\n",
       "month              6.6\n",
       "cons.conf.idx      5.8\n",
       "cons.price.idx     5.5\n",
       "age                4.8\n",
       "job                3.1\n",
       "contact            2.8\n",
       "day_of_week        2.8\n",
       "education          2.3\n",
       "previous           2.2\n",
       "campaign           1.7\n",
       "marital            1.4\n",
       "housing            1.2\n",
       "loan               0.8\n",
       "default            0.7\n",
       "dtype: float64"
      ]
     },
     "execution_count": 29,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "rf = best.named_steps[\"clf\"]\n",
    "names = best.named_steps[\"prep\"].get_feature_names_out()\n",
    "imp = pd.Series(rf.feature_importances_, index=names)\n",
    "orig = {}\n",
    "for n, v in imp.items():\n",
    "    base = n.split(\"__\", 1)[1]\n",
    "    key = next((c for c in cat_features if base.startswith(c + \"_\")), base)\n",
    "    orig[key] = orig.get(key, 0) + v\n",
    "imp_orig = pd.Series(orig).sort_values()\n",
    "fig, ax = plt.subplots(figsize=(8, 6))\n",
    "ax.barh(imp_orig.index, imp_orig.values * 100, color=\"#2e86de\")\n",
    "for i, v in enumerate(imp_orig.values * 100):\n",
    "    ax.text(v + 0.3, i, f\"{v:.1f}%\", va=\"center\", fontsize=9)\n",
    "ax.set_title(f\"{best_name}: Feature Importance by Original Attribute\")\n",
    "ax.set_xlabel(\"Share of total impurity reduction (%)\")\n",
    "ax.set_ylabel(\"Attribute\")\n",
    "savefig(\"cs6_13_feature_importance\")\n",
    "(imp_orig.sort_values(ascending=False) * 100).round(1)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "44090f60",
   "metadata": {},
   "source": [
    "**Interpretation:** the Random Forest relies on the same signals the EDA identified, which supports trusting it:\n",
    "- **Economic indicators** (`euribor3m` 16.0%, `nr.employed` 14.2%, `emp.var.rate` 7.0%, `cons.conf.idx` 5.8%, `cons.price.idx`\n",
    "  5.5%) together account for **≈ 48%** of the model's splitting power.\n",
    "- **Past-campaign history** (`poutcome` 12.2%, `prev_contacted` 8.9%, `previous` 2.2%) adds ≈ 23%.\n",
    "- Timing (`month` 6.6%) and client profile (`age`, `job`, `contact`) contribute the rest.\n",
    "- `housing`, `loan` and `default` are the least important (< 1.3% each) — consistent with housing and loan being statistically\n",
    "  non-significant in Section 2.5.\n",
    "\n",
    "(Impurity importance is biased towards continuous features and splits credit among correlated ones, so these shares are indicative,\n",
    "not exact.)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "037f49b6",
   "metadata": {},
   "source": [
    "### 4.9 Sensitivity check — why `duration` had to be excluded\n",
    "**Purpose:** quantify the leakage the brief warns about by refitting logistic regression *with* `duration` on the same split.\n",
    "This model is **not** a candidate — it only shows how misleading `duration` would be."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 30,
   "id": "92e76ae1",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:31.877156Z",
     "iopub.status.busy": "2026-10-25T17:05:31.876954Z",
     "iopub.status.idle": "2026-10-25T17:05:32.280022Z",
     "shell.execute_reply": "2026-10-25T17:05:32.278043Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Logistic Regression WITHOUT duration: ROC-AUC = 0.8004\n",
      "Logistic Regression WITH duration   : ROC-AUC = 0.9388  (not usable before a call)\n"
     ]
    }
   ],
   "source": [
    "Xd_train = data.loc[X_train.index].drop(columns=[TARGET, \"subscribed\", \"pdays\"])\n",
    "Xd_test = data.loc[X_test.index].drop(columns=[TARGET, \"subscribed\", \"pdays\"])\n",
    "num_d = Xd_train.select_dtypes(include=\"number\").columns.tolist()\n",
    "leaky = Pipeline([(\"prep\", ColumnTransformer([(\"num\", StandardScaler(), num_d),\n",
    "                                              (\"cat\", OneHotEncoder(handle_unknown=\"ignore\"), cat_features)])),\n",
    "                  (\"clf\", LogisticRegression(max_iter=2000, C=0.1))]).fit(Xd_train, y_train)\n",
    "pl = leaky.predict_proba(Xd_test)[:, 1]\n",
    "print(f\"Logistic Regression WITHOUT duration: ROC-AUC = {comparison.loc['Logistic Regression', 'ROC-AUC']:.4f}\")\n",
    "print(f\"Logistic Regression WITH duration   : ROC-AUC = {roc_auc_score(y_test, pl):.4f}  (not usable before a call)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "184c4756",
   "metadata": {},
   "source": [
    "**Interpretation:** adding `duration` lifts logistic regression's ROC-AUC from **0.800 to 0.939** — an apparent jump that is\n",
    "pure leakage. A long call is mostly a *consequence* of the customer being interested (a \"no\" ends the call quickly), and its value\n",
    "is unknown until the call is over. A model reporting 0.94 would look excellent in evaluation and be useless for deciding whom to\n",
    "call. Excluding it, as the brief requires, gives honest performance estimates."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a490884a",
   "metadata": {},
   "source": [
    "## Task 5 — Interpretation and Business Recommendations\n",
    "_The final narrative (best model, suitability, recommendations, limitations) is written in the cells below from the numbers\n",
    "produced above._"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 31,
   "id": "2c81f810",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-10-25T17:05:32.282333Z",
     "iopub.status.busy": "2026-10-25T17:05:32.282104Z",
     "iopub.status.idle": "2026-10-25T17:05:32.321252Z",
     "shell.execute_reply": "2026-10-25T17:05:32.319242Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Accuracy</th>\n",
       "      <th>Precision</th>\n",
       "      <th>Recall</th>\n",
       "      <th>F1-score</th>\n",
       "      <th>ROC-AUC</th>\n",
       "      <th>PR-AUC</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Model</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Logistic Regression</th>\n",
       "      <td>0.8990</td>\n",
       "      <td>0.6633</td>\n",
       "      <td>0.2101</td>\n",
       "      <td>0.3191</td>\n",
       "      <td>0.8004</td>\n",
       "      <td>0.4533</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Decision Tree</th>\n",
       "      <td>0.8970</td>\n",
       "      <td>0.5901</td>\n",
       "      <td>0.2823</td>\n",
       "      <td>0.3819</td>\n",
       "      <td>0.8051</td>\n",
       "      <td>0.4577</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>KNN</th>\n",
       "      <td>0.8991</td>\n",
       "      <td>0.6678</td>\n",
       "      <td>0.2080</td>\n",
       "      <td>0.3172</td>\n",
       "      <td>0.7953</td>\n",
       "      <td>0.4497</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Random Forest</th>\n",
       "      <td>0.9001</td>\n",
       "      <td>0.6606</td>\n",
       "      <td>0.2328</td>\n",
       "      <td>0.3442</td>\n",
       "      <td>0.8114</td>\n",
       "      <td>0.4828</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Random Forest (threshold 0.21)</th>\n",
       "      <td>0.8781</td>\n",
       "      <td>0.4681</td>\n",
       "      <td>0.6013</td>\n",
       "      <td>0.5264</td>\n",
       "      <td>0.8114</td>\n",
       "      <td>0.4828</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                                Accuracy  Precision  Recall  F1-score  \\\n",
       "Model                                                                   \n",
       "Logistic Regression               0.8990     0.6633  0.2101    0.3191   \n",
       "Decision Tree                     0.8970     0.5901  0.2823    0.3819   \n",
       "KNN                               0.8991     0.6678  0.2080    0.3172   \n",
       "Random Forest                     0.9001     0.6606  0.2328    0.3442   \n",
       "Random Forest (threshold 0.21)    0.8781     0.4681  0.6013    0.5264   \n",
       "\n",
       "                                ROC-AUC  PR-AUC  \n",
       "Model                                            \n",
       "Logistic Regression              0.8004  0.4533  \n",
       "Decision Tree                    0.8051  0.4577  \n",
       "KNN                              0.7953  0.4497  \n",
       "Random Forest                    0.8114  0.4828  \n",
       "Random Forest (threshold 0.21)   0.8114  0.4828  "
      ]
     },
     "execution_count": 31,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "summary = comparison[[\"Accuracy\", \"Precision\", \"Recall\", \"F1-score\", \"ROC-AUC\", \"PR-AUC\"]].copy()\n",
    "summary.loc[f\"{best_name} (threshold {best_t:.2f})\"] = [\n",
    "    accuracy_score(y_test, yhat_t), precision_score(y_test, yhat_t), recall_score(y_test, yhat_t),\n",
    "    f1_score(y_test, yhat_t), roc_auc_score(y_test, p), average_precision_score(y_test, p)]\n",
    "summary.round(4)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5122ad07",
   "metadata": {},
   "source": [
    "### 5.1 Best model\n",
    "**Random Forest** (300 trees, max depth 12, min leaf 5) is the recommended model, used with a **decision threshold of 0.21**.\n",
    "\n",
    "| Criterion | Evidence |\n",
    "|---|---|\n",
    "| Ranking quality | Highest ROC-AUC (0.811) and PR-AUC (0.483); reliably above Logistic Regression and KNN (bootstrap CIs exclude 0), narrowly above Decision Tree |\n",
    "| Finding subscribers | At threshold 0.21: recall 0.601, precision 0.468, F1 0.526 (best F1 of any configuration) |\n",
    "| Campaign efficiency | Top 10% of ranked customers contain 46.2% of all subscribers (4.6× lift) |\n",
    "| Sensible drivers | Relies on economic conditions and previous-campaign outcome, matching the EDA and statistical tests |\n",
    "\n",
    "### 5.2 Why it suits the bank's problem\n",
    "- **It ranks customers well**, which is what a call centre needs: a prioritised list, not just yes/no labels. Probabilities let the\n",
    "  bank set the cut-off to match each campaign's call budget.\n",
    "- **It handles the data's structure:** the economic indicators are highly correlated (r up to 0.97) and their effect interacts with\n",
    "  month and contact history. A forest of trees absorbs correlated inputs and non-linear interactions without manual feature\n",
    "  engineering — which a single linear logistic regression cannot.\n",
    "- **It is more stable than a single decision tree:** averaging 300 trees reduces the variance that makes one tree change shape with\n",
    "  small data changes, at the same or better accuracy.\n",
    "- **Trade-off:** it is less transparent than logistic regression and slower to train (~77 s for tuning vs ~4 s). If regulators or\n",
    "  managers require coefficient-level explanations, logistic regression (AUC 0.800) is a close, fully interpretable alternative.\n",
    "\n",
    "### 5.3 Business recommendations\n",
    "1. **Score customers before every campaign and call in ranked order.** Calling the top ~15% of customers (threshold 0.21) reaches\n",
    "   about 60% of all likely subscribers with roughly 47% of calls converting, instead of 11% when calling everyone. With a fixed call\n",
    "   budget, the gains table tells the manager what share of subscribers each budget level reaches.\n",
    "2. **Prioritise high-propensity segments and cap repeat calls.** Customers with a previous campaign success (65% conversion), students\n",
    "   and retirees, and customers reachable by mobile should be contacted first; conversion falls with every repeat contact, so attempts\n",
    "   per customer should be capped at about 3.\n",
    "3. **Adjust campaign intensity to economic conditions** and retrain the model regularly: subscription propensity swings from ~5% to\n",
    "   ~46% with the interest-rate environment, so the model (and the cut-off) must be refreshed as conditions change.\n",
    "\n",
    "### 5.4 Limitations\n",
    "1. **Time dependence of the economic features.** The data spans the 2008–2010 financial crisis, and the economic indicators are the\n",
    "   same for every customer on a given day — so the model partly learns *when* calls succeed, not only *who* subscribes. Because the\n",
    "   80:20 split is random, the training data includes economic periods that overlap with the test data. A time-ordered split (train\n",
    "   on earlier months, test on later ones) would be a stricter test and would likely show lower performance; in a new economic regime\n",
    "   the model must be revalidated before use.\n",
    "2. **Class imbalance and the cost assumption.** With 11% positives, the threshold strongly changes the result. The 0.21 threshold\n",
    "   maximises F1, which weights precision and recall equally; the bank's actual cost of a call versus the profit of a deposit is not\n",
    "   in the data. If those costs were known, the threshold should be chosen to maximise expected profit, which could move it.\n",
    "3. *(Additional)* \"Unknown\" values (20.9% in `default`) were kept as a category; if their meaning differs between campaigns, the\n",
    "   model may learn an unstable pattern.\n",
    "\n",
    "### 5.5 Conclusion\n",
    "Four classifiers were trained on identical, leakage-free inputs and compared on a common test set. All achieved ≈ 90% accuracy — barely\n",
    "above the 88.7% obtained by never predicting a subscription — showing that accuracy cannot guide model choice here. Using ROC-AUC,\n",
    "PR-AUC, recall, F1 and the business cost of missed subscribers, **Random Forest with a 0.21 threshold** is the most useful model: it\n",
    "identifies 60% of subscribers from 14.5% of customers, with nearly half of its calls converting."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.17"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}